# Layer 1 — Record-only baseline（只看单行的基准模型）

## Layer 1 是什么

Dossier（项目技术文档）把整个项目定义为一个**关系信息逐层递增的阶梯（ladder）**：

> Record-only attributes → relationship logic → graph structure → ownership enrichment → whatever else the team can justify

**Layer 1 是阶梯的第一级：只看每一家供应商（third party，第三方）自己的那一行信息。**

- **分析单位**：一条供应商记录（`record_id`）。每家供应商单独作为一个"case"（案例），大小为 1。
- **可以用的信息**：这家供应商**自己的**行，来自所有表（主表、尽调、合同、审批、付款、服务、地址、医疗机构关联），以及参考表（市场、服务分类、员工角色、审批矩阵、国家风险指数、公司事件）。
- **不能用的信息**：**其他供应商的行**。比如"有几家供应商和我共享地址"、"我的 sponsor 名下有几家供应商"，都需要看其他行，属于 Layer 2。
- **判断标准**：在其他所有供应商都不变的情况下，往表里新增一家供应商，Layer 1 变量的值**不应该改变**。

## 为什么需要 Layer 1

Dossier 原话：*"Start from what a record-level view can already find, so that every later gain is measured against it."*

Layer 1 本身不是为了表现好。练习集的 33 个 case 全部是**关系型**异常（共享地址、sponsor 集中、共同受益人……），只看单行**本来就找不全**。Layer 1 的作用是提供一个**对照基准**：后面每加一层关系信息，提升了多少，都和它比。

## 本 notebook 的结构

| Cell | 内容 |
|---|---|
| 1 | Layer 1 是什么（本 cell） |
| 2 | 读数据；列出选用的 67 个变量，以及每个变量来自哪张表的哪一列 |
| 3 | 逐个变量做 data cleaning（数据清洗）并构造变量，得到最终的变量表 `X` |
| 4 | 说明每个需要清洗的变量是怎么清洗的 |
| 5 | Target（目标变量）是什么，risk score（风险分数）是什么 |
| 6 | 建模前对变量做的处理（log、截尾、填补、标准化……），说明 |
| 7 | 上述处理的代码 + 构造 `y`、分组、交叉验证和评估函数 |
| 8–9 | **Model 1**：Linear Regression（线性回归）—— 最直接想到的模型 |
| 10–11 | **Model 2**：Logistic Regression（逻辑回归）—— 修正线性回归输出不是概率的问题 |
| 12–13 | **Model 3**：Decision Tree（决策树）—— 允许阈值和变量交互 |
| 14–15 | **Model 4**：Random Forest（随机森林）—— 很多棵树平均，更稳定 |
| 16–17 | **Model 5**：Isolation Forest（孤立森林）—— 不用标签的无监督模型 |
| 18 | **Model 6**：Multi-level Scorecard（多档计分卡，`model_ms`）—— 传统的分档打分方法；分档规则 |
| 19 | 每个变量的档数和切点（threshold） |
| 20 | 每个变量每一档对应的分数 |
| 21–22 | 权重的计算，以及权重是怎么得到的 |
| 23 | `model_ms`：能否从 1,002 家中把 159 家筛出来 |
| 24–25 | 全部模型对比（Model 1 → Model 6） |
| 26 | 结论 |
| 27 | 导出结果 |

In [1]:
# ============================================================
# Cell 2. 读数据 + 选用的变量清单（来自哪张表、哪一列）
# ============================================================
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option("display.width", 220, "display.max_rows", 200, "display.max_colwidth", 80)

# ---- 数据路径：repo 根目录下的 student_bundle/ -------------------------
# notebook 放在 repo 根目录（Graph-Based-Detection/）时直接可用；
# 放在子文件夹时，自动往上找 student_bundle。
DATA = next(p / "student_bundle" for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "student_bundle").exists())

# ---- 分析时点 t ---------------------------------------------------------
# 练习标签没有日期，所以用快照日；等 validation / test 期标签发布后，
# 把 T 改成对应时期的截止日，所有变量会按 T 重新计算（不会用到未来信息）。
T = pd.Timestamp("2026-06-30")

DATES = {
    "third_parties": ["onboarding_date", "created_date", "terminated_date"],
    "tp_due_diligence": ["dd_completed", "dd_refresh_due"],
    "tp_services": ["valid_from", "valid_to"],
    "tp_engagements": ["start_date", "end_date", "observed_at"],
    "tp_engagement_approvals": ["approval_date", "observed_at"],
    "tp_payments_monthly": ["month"],
    "tp_address_history": ["valid_from", "valid_to", "observed_at"],
    "tp_institution_links": ["valid_from", "valid_to"],
    "healthcare_institutions": [],
    "employee_roles": ["valid_from", "valid_to"],
    "markets": [],
    "service_categories": [],
    "approval_policy_matrix": [],
    "country_risk_context": ["as_of_date", "valid_to"],
    "business_events": ["start_date", "end_date"],
    "labelled_cases_TRAINING_SET": [],
}

def load(name):
    """Prefer parquet (keeps dtypes); fall back to csv."""
    pq, csv = DATA / f"{name}.parquet", DATA / f"{name}.csv"
    try:
        df = pd.read_parquet(pq)
    except Exception:
        df = pd.read_csv(csv)
    for c in DATES[name]:
        df[c] = pd.to_datetime(df[c], errors="coerce")
    return df

tables = {name: load(name) for name in DATES}
for name, df in tables.items():
    print(f"{name:32s} {df.shape[0]:>7,} rows  {df.shape[1]:>3} cols")

# ---- 变量清单 -----------------------------------------------------------
# (变量名, 来源表, 来源列, 中文含义)
# 来源列可以有多个；"ref:" 开头的是参考表（不是这家供应商的行）。
SPEC = [
    # ---------- third_parties（供应商主表） ----------
    ("is_individual", "third_parties", ["party_type"], "是否个人专家"),
    ("expert_hcp_consultant", "third_parties", ["expert_type"], "专家类型：医生顾问"),
    ("expert_investigator_kol", "third_parties", ["expert_type"], "专家类型：临床研究者/KOL"),
    ("expert_regulatory_access", "third_parties", ["expert_type"], "专家类型：注册/准入专家"),
    ("expert_other", "third_parties", ["expert_type"], "专家类型：其他（合并 3 个稀少类别）"),
    ("has_lei", "third_parties", ["lei"], "是否有 LEI 法人识别码"),
    ("has_registry_no", "third_parties", ["registry_no"], "是否有公司注册号"),
    ("remit_country_mismatch", "third_parties", ["remit_to_country", "country"], "收款国 ≠ 所在国"),
    ("region_US", "third_parties", ["country", "ref:markets.region"], "所在大区：美国（参照组=欧洲）"),
    ("region_JAPAN_CHINA", "third_parties", ["country", "ref:markets.region"], "所在大区：日本/中国"),
    ("region_INTL", "third_parties", ["country", "ref:markets.region"], "所在大区：其他国际"),
    ("tenure_years", "third_parties", ["onboarding_date"], "截至 t 的合作年数"),
    ("terminated_by_t", "third_parties", ["terminated_date"], "在 t 时是否已终止"),
    ("migrated_record", "third_parties", ["created_date"], "是否 ERP 迁移记录"),
    ("onboarded_in_event_window", "third_parties", ["onboarding_date", "ref:business_events.start_date", "ref:business_events.end_date"], "是否在收购整合/疫苗上市期间开始合作"),
    ("has_declared_group", "third_parties", ["declared_parent_group"], "是否自报所属集团"),
    ("public_email", "third_parties", ["email_domain"], "是否使用公共邮箱"),
    # ---------- tp_due_diligence（尽调） ----------
    ("q_public_official_interaction", "tp_due_diligence", ["q_public_official_interaction"], "是否代表公司和政府官员打交道"),
    ("dd_sensitive_answer", "tp_due_diligence", ["q_gov_ownership", "q_pep_connected", "q_former_public_official"], "自报国有/PEP相关/前政府官员（任一）"),
    ("adverse_media", "tp_due_diligence", ["adverse_media"], "负面新闻"),
    ("screening_potential_match", "tp_due_diligence", ["screening_result"], "筛查疑似命中（已排除）"),
    ("dd_enhanced", "tp_due_diligence", ["dd_level"], "是否做了加强尽调"),
    ("dd_lead_days", "tp_due_diligence", ["dd_completed", "third_parties.onboarding_date"], "尽调完成到开始合作的天数"),
    ("risk_rating_ord", "tp_due_diligence", ["risk_rating"], "风险评级 0低/1中/2高"),
    # ---------- tp_services（服务类别） ----------
    ("n_service_parents", "tp_services", ["category_code", "taxonomy_version", "ref:service_categories.parent"], "提供几大类服务"),
    ("svc_flag_hcp", "tp_services", ["category_code", "taxonomy_version", "ref:service_categories.flags"], "提供接触医生的服务"),
    ("svc_flag_gov", "tp_services", ["category_code", "taxonomy_version", "ref:service_categories.flags"], "提供接触政府的服务"),
    ("svc_flag_int", "tp_services", ["category_code", "taxonomy_version", "ref:service_categories.flags"], "提供中介类服务"),
    ("svc_flag_flow", "tp_services", ["category_code", "taxonomy_version", "ref:service_categories.flags"], "提供资金/货物流量大的服务"),
    # ---------- tp_engagements（合同） ----------
    ("n_engagements", "tp_engagements", ["engagement_id"], "合同数"),
    ("n_eng_markets", "tp_engagements", ["market"], "合同涉及几个市场"),
    ("n_business_units", "tp_engagements", ["business_unit"], "涉及几个业务部门"),
    ("n_contract_types", "tp_engagements", ["contract_type"], "几种合同类型"),
    ("high_band_share", "tp_engagements", ["value_band"], "高金额档位(3-4)合同比例"),
    ("missing_sponsor_share", "tp_engagements", ["sponsor_employee_id"], "缺发起人的合同比例"),
    ("system_sponsor_share", "tp_engagements", ["sponsor_employee_id", "ref:employee_roles.role"], "发起人为系统默认账号的比例"),
    ("open_engagements", "tp_engagements", ["start_date", "end_date"], "t 时仍在执行的合同数"),
    ("eng_record_lag_days", "tp_engagements", ["start_date", "observed_at"], "合同开始到录入的平均天数"),
    # ---------- tp_engagement_approvals（审批） ----------
    ("missing_approver_share", "tp_engagement_approvals", ["approver_employee_id", "approval_type"], "缺审批人的财务审批比例"),
    ("under_level_approvals", "tp_engagement_approvals", ["approver_employee_id", "approval_type", "tp_engagements.value_band", "ref:employee_roles.approver_level", "ref:approval_policy_matrix.required_approver_level"], "审批人级别低于规定的次数"),
    ("self_approvals", "tp_engagement_approvals", ["approver_employee_id", "tp_engagements.sponsor_employee_id"], "发起人自己审批的次数"),
    ("missing_compliance_reviews", "tp_engagement_approvals", ["approval_type", "tp_engagements.service_category", "ref:service_categories.flags"], "应做合规审查但没做的合同数"),
    ("approval_lead_days", "tp_engagement_approvals", ["approval_date", "tp_engagements.start_date"], "审批到合同开始的平均天数"),
    # ---------- tp_payments_monthly（月度付款） ----------
    ("log_total_usd", "tp_payments_monthly", ["amount_usd"], "付款总额（对数）"),
    ("active_months", "tp_payments_monthly", ["month"], "有付款的月数"),
    ("activity_density", "tp_payments_monthly", ["month", "third_parties.onboarding_date"], "有付款月数 ÷ 合作月数"),
    ("amount_cv", "tp_payments_monthly", ["month", "amount_usd"], "月付款额变异系数"),
    ("round_amount_share", "tp_payments_monthly", ["round_amount_share"], "整数金额发票比例"),
    ("manual_payment_share", "tp_payments_monthly", ["manual_payment_count", "payment_count"], "手工付款比例"),
    ("invoices_per_payment", "tp_payments_monthly", ["invoice_count", "payment_count"], "每次付款对应发票数"),
    ("non_po_invoice_share", "tp_payments_monthly", ["channel"], "无采购订单发票比例"),
    ("p_card_share", "tp_payments_monthly", ["channel"], "公司采购卡付款比例"),
    ("expense_reimbursement_share", "tp_payments_monthly", ["channel"], "员工报销比例"),
    ("n_expense_classes", "tp_payments_monthly", ["expense_classification"], "费用类别数"),
    ("tov_hcp_share", "tp_payments_monthly", ["tov_category"], "向医生/医疗机构利益输送类付款比例"),
    ("sensitive_expense_share", "tp_payments_monthly", ["expense_classification"], "敏感费用比例（佣金/资助捐赠/赞助/政府费用）"),
    # ---------- tp_address_history（地址，只看自己的） ----------
    ("n_addresses", "tp_address_history", ["raw_address"], "地址记录数"),
    ("risky_address_type", "tp_address_history", ["address_type"], "用过注册代理/虚拟办公室/邮政信箱/共享办公"),
    ("org_at_residential", "tp_address_history", ["address_type", "third_parties.party_type"], "公司注册在住宅地址"),
    ("address_type_missing", "tp_address_history", ["address_type"], "地址类型未分类"),
    # ---------- tp_institution_links（医疗机构关联） ----------
    ("n_institution_links", "tp_institution_links", ["institution_id"], "关联机构数"),
    ("public_institution_links", "tp_institution_links", ["institution_id", "ref:healthcare_institutions.public_sector"], "关联公立机构数"),
    ("link_serves", "tp_institution_links", ["link_type"], "作为分销商服务的机构数"),
    ("link_site_of", "tp_institution_links", ["link_type"], "作为临床试验点所属的机构数"),
    ("link_affiliated_with", "tp_institution_links", ["link_type"], "个人专家任职的机构数"),
    # ---------- country_risk_context（国家风险，t 时有效版本） ----------
    ("country_cpi_score", "country_risk_context", ["third_parties.country", "iso2", "index", "as_of_date", "value"], "所在国腐败感知指数 CPI"),
    ("country_bo_access_low", "country_risk_context", ["third_parties.country", "iso2", "index", "as_of_date", "tier"], "所在国受益所有权登记公开程度低"),
]

def col_ref(table, col):
    """'col' -> 'col (#k)', where k is the 1-based column position in its table."""
    if col.startswith("ref:"):
        t, c = col[4:].split(".")
        return f"{t}.{c} (#{tables[t].columns.get_loc(c) + 1})"
    if "." in col:
        t, c = col.split(".")
        return f"{t}.{c} (#{tables[t].columns.get_loc(c) + 1})"
    return f"{col} (#{tables[table].columns.get_loc(col) + 1})"

spec = pd.DataFrame([
    {"variable": v, "source_table": t,
     "source_columns (#column position)": ", ".join(col_ref(t, c) for c in cols),
     "meaning": m}
    for v, t, cols, m in SPEC
])
spec.index = np.arange(1, len(spec) + 1)
VARIABLES = spec["variable"].tolist()

print(f"\nLayer 1 variables: {len(VARIABLES)}")
print(spec.groupby("source_table", sort=False).size().rename("n_variables").to_string())
spec

third_parties                      1,002 rows   16 cols
tp_due_diligence                   1,002 rows   11 cols
tp_services                        1,791 rows    5 cols
tp_engagements                     2,638 rows   12 cols
tp_engagement_approvals            4,298 rows    6 cols
tp_payments_monthly               34,467 rows   18 cols
tp_address_history                 1,091 rows    8 cols
tp_institution_links                 403 rows    6 cols
healthcare_institutions              213 rows    5 cols
employee_roles                     1,502 rows    9 cols
markets                               74 rows    4 cols
service_categories                    43 rows    5 cols
approval_policy_matrix                 5 rows    5 cols
country_risk_context               1,120 rows    8 cols
business_events                        8 rows    5 cols
labelled_cases_TRAINING_SET           33 rows    6 cols

Layer 1 variables: 67
source_table
third_parties              17
tp_due_diligence            7
tp_servi

,variable,source_table,source_columns (#column position),meaning
1,is_individual,third_parties,party_type (#2),是否个人专家
2,expert_hcp_consultant,third_parties,expert_type (#16),专家类型：医生顾问
3,expert_investigator_kol,third_parties,expert_type (#16),专家类型：临床研究者/KOL
4,expert_regulatory_access,third_parties,expert_type (#16),专家类型：注册/准入专家
5,expert_other,third_parties,expert_type (#16),专家类型：其他（合并 3 个稀少类别）
6,has_lei,third_parties,lei (#5),是否有 LEI 法人识别码
7,has_registry_no,third_parties,registry_no (#6),是否有公司注册号
8,remit_country_mismatch,third_parties,"remit_to_country (#15), country (#7)",收款国 ≠ 所在国
9,region_US,third_parties,"country (#7), markets.region (#3)",所在大区：美国（参照组=欧洲）
10,region_JAPAN_CHINA,third_parties,"country (#7), markets.region (#3)",所在大区：日本/中国


In [2]:
# ============================================================
# Cell 3. 逐个变量 data cleaning + 构造，得到最终变量表 X
#   每一步前面的编号 [C1] [C2] ... 对应 Cell 4 的说明
# ============================================================
tp  = tables["third_parties"].copy()
dd  = tables["tp_due_diligence"].copy()
sv  = tables["tp_services"].copy()
en  = tables["tp_engagements"].copy()
ap  = tables["tp_engagement_approvals"].copy()
pm  = tables["tp_payments_monthly"].copy()
ad  = tables["tp_address_history"].copy()
il  = tables["tp_institution_links"].copy()
hi  = tables["healthcare_institutions"]
er  = tables["employee_roles"]
mk  = tables["markets"]
sc  = tables["service_categories"]
apm = tables["approval_policy_matrix"]
cr  = tables["country_risk_context"].copy()
ev  = tables["business_events"]

# ---------------------------------------------------------------
# [C0] 时间截断：只保留在 t 之前发生 且 在 t 之前被记录 的信息
# ---------------------------------------------------------------
tp = tp[tp["onboarding_date"] <= T]
en = en[(en["start_date"] <= T) & (en["observed_at"] <= T)]
ap = ap[(ap["observed_at"] <= T) & ap["engagement_id"].isin(en["engagement_id"])]
pm = pm[(pm["month"] <= T) & pm["engagement_id"].isin(en["engagement_id"])]
sv = sv[sv["valid_from"] <= T]
ad = ad[(ad["valid_from"] <= T) & (ad["observed_at"] <= T)]
il = il[il["valid_from"] <= T]
cr = cr[cr["as_of_date"] <= T]

# [C0b] 布尔列统一成 True/False（csv 读进来可能是字符串）
def to_bool(s):
    return s.map({True: True, False: False, "True": True, "False": False,
                  "true": True, "false": False, 1: True, 0: False}).fillna(False).astype(bool)
for c in ["q_gov_ownership", "q_pep_connected", "q_public_official_interaction",
          "q_former_public_official", "adverse_media"]:
    dd[c] = to_bool(dd[c])
hi = hi.assign(public_sector=to_bool(hi["public_sector"]))

idx = pd.Index(tp["record_id"], name="record_id")
t_ = tp.set_index("record_id")
X = pd.DataFrame(index=idx)

def share(num, den):
    return num / den.replace(0, np.nan)

# ===============================================================
# third_parties
# ===============================================================
X["is_individual"] = (t_["party_type"] == "individual").astype(int)

# [C1] expert_type：公司为空 = "不适用"；3 个稀少类别合并为 expert_other
for e in ["hcp_consultant", "investigator_kol", "regulatory_access"]:
    X[f"expert_{e}"] = (t_["expert_type"] == e).astype(int)
X["expert_other"] = t_["expert_type"].isin(
    ["other_consultant", "patient_expert", "writer_translator"]).astype(int)

# [C2] lei / registry_no：缺失 = 没记录，只做"是否有"
X["has_lei"] = t_["lei"].notna().astype(int)
X["has_registry_no"] = t_["registry_no"].notna().astype(int)

# [C3] 国家代码统一大写、去空格后再比较
country = t_["country"].str.strip().str.upper()
remit = t_["remit_to_country"].str.strip().str.upper()
X["remit_country_mismatch"] = (remit != country).astype(int)

# [C4] country -> region（67 个国家太多，映射成 4 个大区；欧洲为参照组）
region = country.map(mk.set_index("market_id")["region"])
for r in ["US", "JAPAN_CHINA", "INTL"]:
    X[f"region_{r}"] = (region == r).astype(int)

# [C5] 时间类：一律用 onboarding_date；status 不用，按 t 重新计算
X["tenure_years"] = (T - t_["onboarding_date"]).dt.days / 365.25
X["terminated_by_t"] = (t_["terminated_date"] <= T).astype(int)
X["migrated_record"] = (t_["created_date"] == pd.Timestamp("2022-05-01")).astype(int)

# [C6] business_events：收购整合期、疫苗上市期开始合作 -> 可能是良性的集中准入
burst_ev = ev[ev["event_name"].str.contains("acquisition|launch", case=False)]
in_ev = pd.Series(False, index=idx)
for _, e in burst_ev.iterrows():
    end = e["end_date"] if pd.notna(e["end_date"]) else e["start_date"]
    in_ev |= t_["onboarding_date"].between(e["start_date"], end)
X["onboarded_in_event_window"] = in_ev.astype(int)

X["has_declared_group"] = t_["declared_parent_group"].notna().astype(int)

# [C7] email_domain：统一小写，和公共邮箱名单比对
PUBLIC_EMAIL = {"gmail.com", "outlook.com", "yahoo.com", "hotmail.com", "icloud.com",
                "qq.com", "163.com", "aol.com", "gmx.de", "web.de", "yandex.ru", "mail.ru"}
X["public_email"] = t_["email_domain"].str.strip().str.lower().isin(PUBLIC_EMAIL).astype(int)

# ===============================================================
# tp_due_diligence
# ===============================================================
d = dd.set_index("record_id").reindex(idx)
X["q_public_official_interaction"] = d["q_public_official_interaction"].astype(int)
# [C8] 三个极稀少的问卷答案合并成一个
X["dd_sensitive_answer"] = (d["q_gov_ownership"] | d["q_pep_connected"] |
                            d["q_former_public_official"]).astype(int)
X["adverse_media"] = d["adverse_media"].astype(int)
X["screening_potential_match"] = (d["screening_result"] != "clear").astype(int)
X["dd_enhanced"] = (d["dd_level"] == "enhanced").astype(int)
X["dd_lead_days"] = (t_["onboarding_date"] - d["dd_completed"]).dt.days
# [C9] 有序类别 -> 数字
X["risk_rating_ord"] = d["risk_rating"].map({"low": 0, "medium": 1, "high": 2})

# ===============================================================
# [C10] 服务分类统一：v1 / v2 两套代码 -> 通过 service_categories 映射到
#       同一组 parent（大类）和 flags（风险标签）
# ===============================================================
f_v2 = sc.set_index("category_code_v2")["flags"].fillna("")
f_v1 = sc.groupby("category_code_v1")["flags"].apply(lambda s: ";".join(s.fillna("")))
p_v2 = sc.set_index("category_code_v2")["parent"]
p_v1 = sc.groupby("category_code_v1")["parent"].first()

def cat_flags(code, ver):
    return f_v2.get(code, "") if ver == "v2" else f_v1.get(code, "")

def cat_parent(code, ver):
    return p_v2.get(code) if ver == "v2" else p_v1.get(code)

sv["flg"] = [cat_flags(c, v) for c, v in zip(sv["category_code"], sv["taxonomy_version"])]
sv["parent"] = [cat_parent(c, v) for c, v in zip(sv["category_code"], sv["taxonomy_version"])]
g = sv.groupby("record_id")
X["n_service_parents"] = g["parent"].nunique()
for f in ["hcp", "gov", "int", "flow"]:
    X[f"svc_flag_{f}"] = g["flg"].apply(lambda s: s.str.contains(f).any()).astype(float)

# ===============================================================
# tp_engagements
# ===============================================================
en["flg"] = [cat_flags(c, v) for c, v in zip(en["service_category"], en["taxonomy_version"])]
# [C11] 系统默认 sponsor（ERP 迁移时自动分配）单独识别，不当成真实发起人
sys_sponsor = set(er.loc[er["role"] == "system_default", "employee_id"])
en["sys_sponsor"] = en["sponsor_employee_id"].isin(sys_sponsor).astype(int)
en["no_sponsor"] = en["sponsor_employee_id"].isna().astype(int)
# [C12] end_date 为空 = 截至快照仍在执行（不是缺失）
en["open_at_t"] = (en["end_date"].isna() | (en["end_date"] > T)).astype(int)
en["obs_lag"] = (en["observed_at"] - en["start_date"]).dt.days
en["high_band"] = (en["value_band"] >= 3).astype(int)

g = en.groupby("record_id")
X["n_engagements"] = g.size()
X["n_eng_markets"] = g["market"].nunique()
X["n_business_units"] = g["business_unit"].nunique()
X["n_contract_types"] = g["contract_type"].nunique()
X["high_band_share"] = g["high_band"].mean()
X["missing_sponsor_share"] = g["no_sponsor"].mean()
X["system_sponsor_share"] = g["sys_sponsor"].mean()
X["open_engagements"] = g["open_at_t"].sum()
X["eng_record_lag_days"] = g["obs_lag"].mean()

# ===============================================================
# tp_engagement_approvals (+ employee_roles, approval_policy_matrix)
# ===============================================================
lvl = er.set_index("employee_id")["approver_level"]
req = apm.set_index("value_band")["required_approver_level"]
a = ap.merge(en[["engagement_id", "record_id", "value_band", "sponsor_employee_id",
                 "start_date"]], on="engagement_id")
fin = a[a["approval_type"] == "financial"].copy()
fin["no_approver"] = fin["approver_employee_id"].isna().astype(int)
# [C13] 审批人缺失时无法判断级别 -> 不算"级别不够"（NaN < x 为 False）
fin["under_level"] = (fin["approver_employee_id"].map(lvl) <
                      fin["value_band"].map(req)).astype(int)
a["self_approved"] = (a["approver_employee_id"] == a["sponsor_employee_id"]).astype(int)
a["lead_days"] = (a["start_date"] - a["approval_date"]).dt.days
needs_cr = en[en["flg"].str.contains("hcp|gov|int")]
has_cr = set(a.loc[a["approval_type"] == "compliance_review", "engagement_id"])
needs_cr = needs_cr.assign(missing_cr=(~needs_cr["engagement_id"].isin(has_cr)).astype(int))

X["missing_approver_share"] = fin.groupby("record_id")["no_approver"].mean()
X["under_level_approvals"] = fin.groupby("record_id")["under_level"].sum()
X["self_approvals"] = a.groupby("record_id")["self_approved"].sum()
X["missing_compliance_reviews"] = needs_cr.groupby("record_id")["missing_cr"].sum()
X["approval_lead_days"] = a.groupby("record_id")["lead_days"].mean()

# ===============================================================
# tp_payments_monthly
# ===============================================================
pm["tov_hcp"] = (pm["tov_category"] != "not_applicable").astype(int)
pm["sensitive_exp"] = pm["expense_classification"].isin(
    ["commissions", "grants_donations", "sponsorships", "government_fees"]).astype(int)
for ch in ["non_po_invoice", "p_card", "expense_reimbursement"]:
    pm[f"ch_{ch}"] = (pm["channel"] == ch).astype(int)

g = pm.groupby("record_id")
# [C14] 金额右偏严重 -> 取 log1p
X["log_total_usd"] = np.log1p(g["amount_usd"].sum())
X["active_months"] = g["month"].nunique()
X["activity_density"] = share(X["active_months"], X["tenure_years"] * 12).clip(upper=1)
# [C15] 先按月汇总，再算波动；只有 1 个月的供应商 std 无定义 -> 缺失
monthly = pm.groupby(["record_id", "month"])["amount_usd"].sum().groupby("record_id")
X["amount_cv"] = monthly.std() / monthly.mean()
X["round_amount_share"] = g["round_amount_share"].mean()
X["manual_payment_share"] = share(g["manual_payment_count"].sum(), g["payment_count"].sum())
X["invoices_per_payment"] = share(g["invoice_count"].sum(), g["payment_count"].sum())
for ch in ["non_po_invoice", "p_card", "expense_reimbursement"]:
    X[f"{ch}_share"] = g[f"ch_{ch}"].mean()
X["n_expense_classes"] = g["expense_classification"].nunique()
X["tov_hcp_share"] = g["tov_hcp"].mean()
X["sensitive_expense_share"] = g["sensitive_exp"].mean()

# ===============================================================
# tp_address_history（只看自己的地址类型，不比较其他供应商）
# ===============================================================
# [C16] address_type 统一小写；缺失单独标记，不当成"安全"
ad["address_type"] = ad["address_type"].str.strip().str.lower()
ad["risky_type"] = ad["address_type"].isin(
    ["registered_agent", "virtual_office", "po_box", "coworking"]).astype(int)
ad["residential"] = (ad["address_type"] == "residential").astype(int)
ad["type_missing"] = ad["address_type"].isna().astype(int)
g = ad.groupby("record_id")
X["n_addresses"] = g.size()
X["risky_address_type"] = g["risky_type"].max()
X["org_at_residential"] = g["residential"].max() * (1 - X["is_individual"])
X["address_type_missing"] = g["type_missing"].max()

# ===============================================================
# tp_institution_links (+ healthcare_institutions)
# ===============================================================
il = il.merge(hi[["institution_id", "public_sector"]], on="institution_id", how="left")
g = il.groupby("record_id")
X["n_institution_links"] = g.size()
X["public_institution_links"] = g["public_sector"].sum()
for lt in ["SERVES", "SITE_OF", "AFFILIATED_WITH"]:
    X[f"link_{lt.lower()}"] = il[il["link_type"] == lt].groupby("record_id").size()

# ===============================================================
# [C17] country_risk_context：取 t 时最新生效的版本
# ===============================================================
def latest(index_name, col):
    s = cr[cr["index"] == index_name].sort_values("as_of_date").groupby("iso2")[col].last()
    return country.map(s)

X["country_cpi_score"] = latest("CPI", "value")
X["country_bo_access_low"] = (latest("BO_REGISTER_ACCESS", "tier") == "low").astype(int)

# ===============================================================
# [C18] 缺失值处理
#   (a) 计数类：没有对应的行 = 0 次（真的没有）
#   (b) 比例/均值类：没有分母 = 无定义 -> 保持 NaN，建模时用中位数填补
# ===============================================================
COUNT_VARS = ["n_service_parents", "n_engagements", "n_eng_markets", "n_business_units",
              "n_contract_types", "open_engagements", "under_level_approvals",
              "self_approvals", "missing_compliance_reviews", "active_months",
              "n_expense_classes", "n_addresses", "n_institution_links",
              "public_institution_links", "link_serves", "link_site_of",
              "link_affiliated_with", "svc_flag_hcp", "svc_flag_gov", "svc_flag_int",
              "svc_flag_flow", "risky_address_type", "org_at_residential"]
X[COUNT_VARS] = X[COUNT_VARS].fillna(0)

X = X[VARIABLES].astype(float)          # 列顺序和 Cell 2 的清单一致

# ---- 检查 ------------------------------------------------------------
assert list(X.columns) == VARIABLES and X.index.is_unique
check = pd.DataFrame({
    "missing_share": X.isna().mean().round(3),
    "n_unique": X.nunique(),
    "min": X.min().round(2),
    "median": X.median().round(2),
    "max": X.max().round(2),
})
print(f"X: {X.shape[0]} records x {X.shape[1]} variables (as of {T.date()})")
print("variables with missing values (filled with median at model time):")
print(check[check["missing_share"] > 0].to_string())
check

X: 1002 records x 67 variables (as of 2026-06-30)
variables with missing values (filled with median at model time):
                             missing_share  n_unique   min  median    max
high_band_share                      0.015        22  0.00    0.00   1.00
missing_sponsor_share                0.015        17  0.00    0.00   1.00
system_sponsor_share                 0.015         9  0.00    0.00   1.00
eng_record_lag_days                  0.015       106  0.00    2.50  22.00
missing_approver_share               0.015        19  0.00    0.00   1.00
approval_lead_days                   0.015       258  3.00   17.17  33.50
log_total_usd                        0.017       985  5.22   12.76  20.42
activity_density                     0.018       863  0.01    0.35   1.00
amount_cv                            0.029       973  0.01    0.52   2.26
round_amount_share                   0.017       940  0.01    0.08   0.23
manual_payment_share                 0.017       294  0.00    0.01   0

,missing_share,n_unique,min,median,max
is_individual,0.000,2,0.00,0.00,1.00
expert_hcp_consultant,0.000,2,0.00,0.00,1.00
expert_investigator_kol,0.000,2,0.00,0.00,1.00
expert_regulatory_access,0.000,2,0.00,0.00,1.00
expert_other,0.000,2,0.00,0.00,1.00
has_lei,0.000,2,0.00,0.00,1.00
has_registry_no,0.000,2,0.00,0.00,1.00
remit_country_mismatch,0.000,2,0.00,0.00,1.00
region_US,0.000,2,0.00,0.00,1.00
region_JAPAN_CHINA,0.000,2,0.00,0.00,1.00


## Cell 4 · Data cleaning（数据清洗）说明

编号对应 Cell 3 代码里的 `[C*]`。只列需要清洗或有特殊处理的变量；其余变量是直接取值或简单计数。

| 编号 | 涉及变量 | 原始问题 | 处理方法 |
|---|---|---|---|
| **C0** | 所有变量 | 数据有两种时间：*valid time*（有效时间，事情实际发生的时间）和 *observed time*（观测时间，公司记录下来的时间）。只按一种过滤会用到未来信息（data leakage，数据泄漏） | 只保留 **发生在 t 之前、且在 t 之前被记录** 的行 |
| **C0b** | 问卷答案、adverse_media、public_sector | 从 csv 读进来时布尔值可能是字符串 `"True"` | 统一转成 True/False |
| **C1** | expert_* | ① 公司的 expert_type 为空，意思是"不适用"而不是缺失；② patient_expert（8）、writer_translator（5）太稀少 | ① 公司在所有 expert_* 上都为 0；② 把 other_consultant、patient_expert、writer_translator 合并成 `expert_other`（共 24 人） |
| **C2** | has_lei, has_registry_no | LEI 缺 67%、注册号缺 64%；缺失是"没记录"，不是"没有" | 不用原值，只做"是否有"的 0/1 |
| **C3** | remit_country_mismatch | 国家代码可能有大小写、空格差异 | 统一大写、去空格后再比较 |
| **C4** | region_* | 67 个国家，很多国家只有几家供应商，直接 one-hot（独热编码）太稀疏 | 通过 `markets.csv` 映射成 4 个大区；欧洲作为参照组（不建列），避免和截距完全共线 |
| **C5** | tenure_years, terminated_by_t, migrated_record | ① 169 条记录的 `created_date` 都是 2022-05-01（ERP 系统迁移重新录入），不能代表合作开始；② `status` 是快照日 2026-06-30 的状态，用它等于知道未来 | ① 合作时长一律用 `onboarding_date`；迁移记录单独做 0/1；② 不用 `status`，改为按 t 计算"`terminated_date` ≤ t" |
| **C6** | onboarded_in_event_window | 收购整合、疫苗上市会让一批供应商集中开始合作，属于良性情况 | 用 `business_events.csv` 的时间窗口标记 |
| **C7** | public_email | 268 条邮箱域名重复，大多是 gmail、outlook 等公共邮箱 | 小写后与公共邮箱名单比对；域名原值不当变量（留给 Layer 2 做连接） |
| **C8** | dd_sensitive_answer | q_gov_ownership（4 个 True）、q_pep_connected（5）、q_former_public_official（3）单独都太稀少 | 合并为"三者任一为 True" |
| **C9** | risk_rating_ord | 文字类别 low / medium / high | 有序映射为 0 / 1 / 2；`dd_level` 和它几乎完全重合，只保留"是否加强尽调"0/1 |
| **C10** | n_service_parents, svc_flag_* | 2024-01-01 服务分类重新编码，v1、v2 混用；v1 → v2 是一对多 | 用 `service_categories.csv` 把两套代码都映射到同一组 parent（大类）和 flags（风险标签）；v1 的一个代码对应多个 v2 时，取它们 flags 的并集 |
| **C11** | system_sponsor_share | ERP 迁移时，没有 sponsor 的合同被分给一个"系统默认账号"，看起来像真实发起人 | 通过 `employee_roles` 中 role = `system_default` 识别并单独计算比例 |
| **C12** | open_engagements | `end_date` 为空占 26% | 为空 = 截至快照仍在执行（数据字典的定义），不是缺失 |
| **C13** | under_level_approvals | 审批人缺失 13.7%，缺失时无法知道级别 | 审批人缺失的那次审批不计入"级别不够"；缺失本身由 `missing_approver_share` 单独记录 |
| **C14** | log_total_usd | 付款总额严重右偏（少数供应商金额极大） | 取 log(1 + x) |
| **C15** | amount_cv | 原表是"供应商 × 合同 × 月 × 费用类型 × 渠道"，同一个月有多行 | 先按月汇总再算变异系数；只有 1 个月付款的供应商无法计算，留为缺失 |
| **C16** | risky_address_type, org_at_residential, address_type_missing | ① 大小写可能不一致；② `address_type` 缺 38% | ① 统一小写；② 缺失**不当成安全地址**，单独做 `address_type_missing` |
| **C17** | country_cpi_score, country_bo_access_low | CPI 每年一个版本，不能用最新版去分析过去 | 取 `as_of_date` ≤ t 的最新版本 |
| **C18** | 所有聚合变量 | 一家供应商在某张表里没有行时，聚合结果为空 | **计数类**：没有行 = 0（真的没有）；**比例/均值类**：没有分母 = 无定义，保留缺失，建模时用中位数填补 |

### 设计时就排除、没有进入变量清单的列

| 列 | 原因 |
|---|---|
| record_id、legal_name、tax_id、phone、email_domain 原值、raw_address、所有 employee_id | ID 或联系方式本身没有含义；它们的作用是**连接不同供应商**，属于 Layer 2 |
| status | 快照日状态，会泄漏未来信息（已由 terminated_by_t 替代） |
| dd_level | 和 risk_rating 几乎完全重合 |
| dd_refresh_due | 只记录了第一次尽调，无法判断后来是否逾期 |
| address_role | 全部是 registered，没有信息量 |
| max_value_band、max_invoice_band、cross_border_*、n_paying_entities、burstiness | 和已保留变量高度相关（|ρ| ≥ 0.9） |
| fmv 超上限、发票超档位、所在国受制裁 | 少于 10 家供应商有这种情况，太稀少 |

## Cell 5 · Target（目标变量）和 risk score（风险分数）

### 1. 分析单位：一家供应商 = 一个 case

Layer 1 只看单行，没有任何关系信息，所以**无法把几家供应商归成一组**（归组本身需要"共享地址、共享 sponsor"这类关系，那是 Layer 2）。因此在 Layer 1 中，**每家供应商单独是一个大小为 1 的 case**。

这也是统一框架的最简单情况：后面每一层都是"先生成候选 case → 再给 case 打风险分"，只是候选 case 越来越大、case 变量越来越丰富。

### 2. Target $y$

$$
y_i = \begin{cases} 1 & \text{供应商 } i \text{ 是 labelled\_cases\_TRAINING\_SET 中任一 case 的成员} \\ 0 & \text{否则} \end{cases}
$$

- 33 个标记 case，展开后 159 家供应商为 1，占 15.9%。
- 当 case 大小为 1 时，"这个 case 是否和某个标记 case 匹配"就等价于"这家供应商是否属于某个标记 case"，所以 case 层面的 target 退化为上式。

### 3. Risk score（风险分数）是模型的**输出**，不是 target

$$
\text{risk}_i = f(x_i)
$$

- **有监督模型**（Model 1–4：线性回归、逻辑回归、决策树、随机森林）：用 $y$ 训练，$\text{risk}_i$ 是预测的 $y_i = 1$ 的可能性。
- **无监督模型**（Model 5：孤立森林）：$\text{risk}_i$ = 异常程度，训练时**不看 $y$**，$y$ 只用于评估。之所以必须做无监督：主数据集约 2 万条，**没有标签**；validation（验证期）标签项目中期才发布；test（测试期）由 GSK 盲评。
- Dossier 提醒：无监督分数是**排名**，不是概率，除非校准过。

### 4. 为什么不用 `risk_rating`（风险评级）当 target

`risk_rating` 是公司 onboarding 时用**现有规则**打的分。拿它当 target，模型只会学着模仿现有规则；而本项目要找的恰恰是现有规则发现不了的问题。而且它已经是输入变量之一，不能同时当输入和答案。

### 5. 两个需要控制的问题

- **同一个 case 的成员不能拆开放。** 交叉验证用 **StratifiedGroupKFold（分层分组交叉验证）**：同一个 case 的成员一定在同一折，否则模型会"见过答案的一部分"。有 2 家供应商同时属于两个 case，这两个 case 合并成一组。
- **Shortcut（捷径）变量。** 部分变量在练习集上区分力很强，但很可能是练习集的特点而不是真规律：地区和 CPI（练习 case 集中在国际市场）、终止状态和合作年数（练习 case 集中在近几年）、风险评级（公司已有判断）。所以跑两套变量：
  - **Full（完整）**：67 个变量
  - **No-shortcut（去捷径）**：去掉上述 7 个变量，剩 60 个

### 6. 评估指标

分析员只看排在最前面的 k 个，正负例又不平衡，所以按 dossier 要求看**排名质量**，不看 accuracy（准确率）：

| 指标 | 含义 |
|---|---|
| Precision@k | 前 k 名里有多少是真的 |
| Recall@k | 所有正例中有多少进了前 k 名 |
| Average Precision（AP，平均精度） | 综合整条排名的精度 |
| Case hit@k | 33 个标记 case 中，有多少个**至少一个成员**进了前 k 名 |
| Member recall@k | 平均每个 case 有多大比例的成员进了前 k 名 |

## Cell 6 · 建模前对变量做的处理（代码在 Cell 7）

Cell 3 的清洗解决的是"数据本身的问题"（格式、版本、缺失的含义、时间泄漏）。这一步解决的是"**变量放进模型之前的形状问题**"。不同模型需要的处理不一样：线性类模型（线性回归、逻辑回归）对变量的尺度和极端值很敏感；树类模型（决策树、随机森林）只看变量的**排序**，所以基本不需要处理。

### 6.1 各项处理

| # | 处理 | 涉及变量 | 为什么 | 用于哪些模型 |
|---|---|---|---|---|
| T1 | **log(1 + x) 变换** | 付款总额（Cell 3 已取对数）；另外 14 个右偏的计数/天数变量：`n_engagements`、`open_engagements`、`active_months`、`n_eng_markets`、`n_addresses`、`n_institution_links`、`link_serves`、`link_site_of`、`public_institution_links`、`under_level_approvals`、`self_approvals`、`missing_compliance_reviews`、`dd_lead_days`、`eng_record_lag_days` | 这些变量偏度（skewness）都大于 2，比如 `link_serves` 偏度 15.3：99% 的供应商 ≤ 6，最大值却是 47。不取对数时，一两个极端值会主导线性模型的系数。取 log(1+x) 是因为有 0 值（log 0 无定义） | 线性回归、逻辑回归 |
| T2 | **winsorize（截尾）到第 1 和第 99 百分位** | 所有连续变量 | 进一步压住极端值，比如 `dd_lead_days` 的 99% 分位是 309 天，最大值 778 天。**截尾的上下界只用训练折计算**，再套用到测试折，避免用到测试数据的信息 | 线性回归、逻辑回归 |
| T3 | **缺失值：中位数填补 + 缺失指示变量** | 17 个比例/均值类变量（缺失 1.5%–2.9%） | 缺失的都是没有合同或没有付款记录的供应商（约 15–29 家），比例无定义。用中位数填补的同时，额外加一列"是否缺失"0/1，让模型自己判断"没有付款记录"本身是否有信息。中位数同样只用训练折计算 | 所有模型 |
| T4 | **标准化（z-score）**：$z = (x - \bar x)/s$ | 所有变量 | 让系数可以互相比较：系数表示"变量增加 1 个标准差，分数变化多少" | 线性回归、逻辑回归、孤立森林 |
| T5 | **类别不平衡加权**：`class_weight="balanced"` | target $y$ | 正例只占 15.9%。不加权时模型会偏向"全部预测为 0"。加权后正例的权重约为负例的 5.3 倍（= 843 / 159） | 逻辑回归、决策树、随机森林 |
| T6 | **两套变量** | Full 67 个 / No-shortcut 60 个 | 去掉 7 个可能是练习集捷径的变量（地区 3 个、CPI、终止状态、合作年数、风险评级），检查模型是否靠它们拿分 | 所有模型 |

**树类模型为什么不做 T1、T2、T4：** 树的每次分裂都是"变量 ≤ 某个值"。log 变换、截尾、标准化都不改变数值的大小顺序，所以树的结果不变，做了也没有意义。

### 6.2 交叉验证（cross-validation）

- **StratifiedGroupKFold，5 折**：同一个标记 case 的成员一定在同一折（Cell 5 说明过原因），并且每折的正例比例大致相同。
- 每家供应商的分数都来自**没见过它**的模型（out-of-fold prediction，折外预测）。
- 重复 **5 个随机种子**（seed），报告平均值和标准差，检查结果是否稳定，而不是一次运气好。
- 排名并列时随机打破平局（决策树的分数只有少数几个取值，并列很多）。

In [3]:
# ============================================================
# Cell 7. y、分组、变量处理、交叉验证和评估函数
# ============================================================
from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import average_precision_score

# ---------- target y ----------
cases = tables["labelled_cases_TRAINING_SET"].copy()
cases["members"] = cases["member_record_ids"].str.split("|")
membership = cases[["case_id", "members"]].explode("members").rename(columns={"members": "record_id"})
membership = membership[membership["record_id"].isin(X.index)]
y = pd.Series(X.index.isin(membership["record_id"]).astype(int), index=X.index, name="y")

# ---------- groups：同一 case 同组；共享成员的 case 合并 ----------
parent = {}
def find(a):
    parent.setdefault(a, a)
    while parent[a] != a:
        parent[a] = parent[parent[a]]
        a = parent[a]
    return a
for _, grp in membership.groupby("record_id")["case_id"]:
    ids = grp.tolist()
    for other in ids[1:]:
        parent[find(ids[0])] = find(other)
case_group = (membership.assign(group=membership["case_id"].map(find))
              .drop_duplicates("record_id").set_index("record_id")["group"])
rid = pd.Series(X.index, index=X.index)
groups = rid.map(case_group).fillna(rid)
print(f"records {len(y)}, positives {y.sum()} ({y.mean():.1%}), "
      f"CV groups among positives {groups[y == 1].nunique()}")

# ---------- T1: log(1+x) ----------
LOG_VARS = ["n_engagements", "open_engagements", "active_months", "n_eng_markets",
            "n_addresses", "n_institution_links", "link_serves", "link_site_of",
            "public_institution_links", "under_level_approvals", "self_approvals",
            "missing_compliance_reviews", "dd_lead_days", "eng_record_lag_days"]
X_lin = X.copy()
X_lin[LOG_VARS] = np.log1p(X_lin[LOG_VARS])
skew = pd.DataFrame({"skew_before": X[LOG_VARS].skew(), "skew_after_log": X_lin[LOG_VARS].skew()}).round(2)
print("\nT1 log(1+x):"); print(skew.to_string())

# ---------- T2: winsorize（只用训练折的分位数）----------
class Winsorizer(BaseEstimator, TransformerMixin):
    def __init__(self, lower=0.01, upper=0.99):
        self.lower, self.upper = lower, upper
    def fit(self, Z, y=None):
        self.lo_ = np.nanquantile(Z, self.lower, axis=0)
        self.hi_ = np.nanquantile(Z, self.upper, axis=0)
        return self
    def transform(self, Z):
        return np.clip(Z, self.lo_, self.hi_)

# 线性类：T3 填补 + 缺失指示 → T2 截尾 → T4 标准化
def linear_prep():
    return [SimpleImputer(strategy="median", add_indicator=True), Winsorizer(), StandardScaler()]
# 树类：只做 T3
def tree_prep():
    return [SimpleImputer(strategy="median", add_indicator=True)]

# ---------- T6: 两套变量 ----------
SHORTCUT_VARS = ["region_US", "region_JAPAN_CHINA", "region_INTL", "country_cpi_score",
                 "terminated_by_t", "tenure_years", "risk_rating_ord"]
FEATURE_SETS = {"Full": VARIABLES,
                "No-shortcut": [v for v in VARIABLES if v not in SHORTCUT_VARS]}
SEEDS = [0, 1, 2, 3, 4]

def oof_scores(model, Xs, seed):
    """5 折分组交叉验证的折外分数（有监督模型用）。"""
    cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=seed)
    score = pd.Series(np.nan, index=Xs.index)
    for tr, te in cv.split(Xs, y, groups):
        m = clone(model).fit(Xs.iloc[tr], y.iloc[tr])
        score.iloc[te] = (m.predict_proba(Xs.iloc[te])[:, 1] if hasattr(m, "predict_proba")
                          else m.predict(Xs.iloc[te]))
    return score

# ---------- 评估 ----------
KS = [10, 25, 50, 100]
case_members = cases.set_index("case_id")["members"].apply(lambda m: [r for r in m if r in X.index])

def evaluate(score, seed=0):
    tie = np.random.default_rng(seed).random(len(score))          # 随机打破并列
    order = score.index[np.lexsort((tie, -score.to_numpy()))]
    yy = y.loc[order].to_numpy()
    out = {"AP": average_precision_score(y, score.loc[y.index])}
    for k in KS:
        out[f"P@{k}"] = yy[:k].mean()
    out["R@100"] = yy[:100].sum() / yy.sum()
    out["first_hit"] = int(np.argmax(yy == 1) + 1)
    top = set(order[:100])
    out["case_hit@100"] = case_members.apply(lambda m: any(r in top for r in m)).mean()
    out["member_recall@100"] = case_members.apply(lambda m: np.mean([r in top for r in m])).mean()
    return out

RESULTS, SCORES = [], {}

def run(name, model, X_source, supervised=True):
    """对两套变量 × 5 个 seed 打分并评估；返回 (均值, 标准差) 表。"""
    rows = []
    for fs, cols in FEATURE_SETS.items():
        per_seed = []
        for s in SEEDS:
            m = clone(model)
            if "random_state" in m.get_params():
                m.set_params(random_state=s)
            for k in m.get_params():                     # pipeline 内的随机种子
                if k.endswith("__random_state"):
                    m.set_params(**{k: s})
            if supervised:
                sc_ = oof_scores(m, X_source[cols], s)
            else:
                sc_ = pd.Series(m.fit(X_source[cols]).score_samples(X_source[cols]) * -1,
                                index=X_source.index)
            per_seed.append(sc_)
            rows.append({"model": name, "features": fs, "seed": s, **evaluate(sc_, s)})
        SCORES[(name, fs)] = pd.concat(per_seed, axis=1).mean(axis=1)
    df = pd.DataFrame(rows)
    RESULTS.append(df)
    g = df.drop(columns=["model", "seed"]).groupby("features", sort=False)
    return g.mean().round(3), g.std().round(3)

# 随机排序基准
rand_rows = [{"model": "Random", "features": fs, "seed": s,
              **evaluate(pd.Series(np.random.default_rng(100 + s).random(len(y)), index=y.index), s)}
             for fs in FEATURE_SETS for s in SEEDS]
RESULTS.append(pd.DataFrame(rand_rows))
print(f"\nrandom-ranking precision = positive rate = {y.mean():.3f}")

records 1002, positives 159 (15.9%), CV groups among positives 31

T1 log(1+x):
                            skew_before  skew_after_log
n_engagements                      1.72            0.16
open_engagements                   2.62            1.58
active_months                      1.54           -0.61
n_eng_markets                      3.86            0.53
n_addresses                        3.01            2.63
n_institution_links               14.87            3.57
link_serves                       15.30            5.43
link_site_of                       8.59            4.76
public_institution_links           8.16            3.33
under_level_approvals              9.64            6.31
self_approvals                     4.91            4.42
missing_compliance_reviews         3.80            2.91
dd_lead_days                       8.40            0.31
eng_record_lag_days                2.47            0.26

random-ranking precision = positive rate = 0.159


## Cell 8 · Model 1：Linear Regression（线性回归）

### 为什么从它开始

这是拿到"一堆 $x$ 和一个 $y$"时**最直接想到的模型**：把 $y$ 写成 $x$ 的加权和，

$$
\hat y_i = \beta_0 + \beta_1 x_{i1} + \beta_2 x_{i2} + \cdots + \beta_p x_{ip}
$$

用最小二乘法（OLS，ordinary least squares）估计 $\beta$。$y$ 是 0/1 时，这个模型也叫 **linear probability model（线性概率模型）**：$\hat y_i$ 被当成"属于标记 case 的可能性"，直接拿来排序。

（严格来说 *simple* linear regression 指只有一个自变量；这里用全部变量，是 *multiple* linear regression，多元线性回归。）

### 怎么设

| 项目 | 设置 |
|---|---|
| 输入 | `X_lin`（做过 T1 log 变换）→ T3 填补 + 缺失指示 → T2 截尾 → T4 标准化 |
| 模型 | `LinearRegression()`，无正则化 |
| 输出（risk score） | $\hat y_i$，按从大到小排序 |
| 训练/评估 | 5 折分组交叉验证 × 5 个 seed，两套变量 |

### 预期的问题（也是换下一个模型的理由）

1. **$\hat y$ 不在 [0, 1] 内**：直线没有上下限，会出现负的"概率"或大于 1 的"概率"。排序还能用，但不能解释成概率。
2. **误差不符合假设**：$y$ 只有 0 和 1 两个值，残差的方差随 $\hat y$ 变化（heteroskedasticity，异方差），OLS 的标准误和显著性检验都不可靠。
3. **只能表示"越大越危险"或"越大越安全"的直线关系**，表示不了阈值（比如"只要出现过一次就很可疑"）和变量之间的交互。

In [4]:
# ============================================================
# Cell 9. model_linear
# ============================================================
from sklearn.linear_model import LinearRegression

model_linear = make_pipeline(*linear_prep(), LinearRegression())
mean_linear, std_linear = run("1 Linear regression", model_linear, X_lin)
print("mean over 5 seeds"); print(mean_linear.to_string())
print("\nstd over 5 seeds"); print(std_linear[["AP", "P@50", "R@100"]].to_string())

# ---- 问题 1：预测值超出 [0,1] ----
cols = FEATURE_SETS["No-shortcut"]
fit_linear = clone(model_linear).fit(X_lin[cols], y)
pred = fit_linear.predict(X_lin[cols])
print(f"\npredictions < 0: {(pred < 0).sum()},  > 1: {(pred > 1).sum()}  "
      f"(range {pred.min():.2f} to {pred.max():.2f})")

# ---- 系数（标准化后，No-shortcut，全部数据拟合，仅用于解释）----
names = list(cols) + [f"missing:{c}" for c in np.array(cols)[fit_linear[0].indicator_.features_]]
coef_linear = pd.Series(fit_linear[-1].coef_, index=names).sort_values()
print("\nlargest coefficients (per 1 SD):")
print(pd.concat([coef_linear.head(5), coef_linear.tail(8)]).round(3).to_string())

mean over 5 seeds
                AP  P@10   P@25   P@50  P@100  R@100  first_hit  case_hit@100  member_recall@100
features                                                                                        
Full         0.589   1.0  0.992  0.796  0.638  0.401        1.0         0.679              0.420
No-shortcut  0.487   1.0  1.000  0.892  0.556  0.350        1.0         0.527              0.386

std over 5 seeds
                AP   P@50  R@100
features                        
Full         0.025  0.048  0.019
No-shortcut  0.020  0.030  0.014

predictions < 0: 149,  > 1: 19  (range -0.21 to 1.32)

largest coefficients (per 1 SD):
link_serves                -0.311
link_site_of               -0.189
link_affiliated_with       -0.188
high_band_share            -0.059
n_engagements              -0.055
expert_hcp_consultant       0.021
dd_enhanced                 0.024
log_total_usd               0.030
activity_density            0.054
sensitive_expense_share     0.054
missing:activit

## Cell 10 · Model 2：Logistic Regression（逻辑回归）

### 为什么可以用，以及它改进了什么

针对 Model 1 的问题 1 和 2：$y$ 是 0/1，应该直接对**概率**建模。逻辑回归仍然是线性组合，但通过 sigmoid（S 形）函数把它压到 (0, 1)：

$$
\hat P(y_i = 1 \mid x_i) = \frac{1}{1 + e^{-(\beta_0 + \beta^\top x_i)}}
\qquad\Longleftrightarrow\qquad
\log \frac{P}{1 - P} = \beta_0 + \beta^\top x_i
$$

- 输出一定在 0 到 1 之间，可以解释为概率。
- 用最大似然估计，假设就是 $y$ 服从伯努利分布，和 0/1 的数据一致。
- **系数仍然好解释**：$e^{\beta_j}$ 是 odds ratio（优势比），即 $x_j$ 增加 1 个标准差时，"是正例 vs 不是正例"的比值乘以多少。这对给分析员写 reason codes（打分理由）很重要。

### 怎么设

| 项目 | 设置 | 理由 |
|---|---|---|
| 输入 | 和 Model 1 完全相同 | 只换模型，才能看出改进来自模型本身 |
| 正则化 | L2，`C = 0.1` | 67 个变量、159 个正例，变量多样本少，不加正则化系数会很不稳定；L2 把系数往 0 收缩 |
| 类别加权 | `class_weight="balanced"`（T5） | 正例只占 15.9% |
| 输出 | $\hat P(y_i = 1)$ | |

### 仍然存在的限制

和 Model 1 一样，它对每个变量只能是**单调的直线关系**（在 log-odds 尺度上），表示不了阈值和交互，比如"敏感费用比例高 **并且** 合同涉及多个部门"。

In [5]:
# ============================================================
# Cell 11. model_logistic
# ============================================================
from sklearn.linear_model import LogisticRegression

model_logistic = make_pipeline(*linear_prep(),
                               LogisticRegression(C=0.1, class_weight="balanced", max_iter=5000))
mean_logistic, std_logistic = run("2 Logistic regression", model_logistic, X_lin)
print("mean over 5 seeds"); print(mean_logistic.to_string())
print("\nstd over 5 seeds"); print(std_logistic[["AP", "P@50", "R@100"]].to_string())

# ---- 系数和 odds ratio（No-shortcut，全部数据拟合，仅用于解释）----
cols = FEATURE_SETS["No-shortcut"]
fit_logistic = clone(model_logistic).fit(X_lin[cols], y)
names = list(cols) + [f"missing:{c}" for c in np.array(cols)[fit_logistic[0].indicator_.features_]]
coef_logistic = pd.Series(fit_logistic[-1].coef_[0], index=names).sort_values()
tbl = pd.DataFrame({"coef (per 1 SD)": coef_logistic,
                    "odds ratio": np.exp(coef_logistic)}).round(3)
print("\nlargest coefficients:")
print(pd.concat([tbl.head(5), tbl.tail(8)]).to_string())

mean over 5 seeds
                AP  P@10   P@25   P@50  P@100  R@100  first_hit  case_hit@100  member_recall@100
features                                                                                        
Full         0.602   1.0  0.976  0.808  0.660  0.415        1.0         0.691              0.412
No-shortcut  0.462   1.0  1.000  0.812  0.504  0.317        1.0         0.533              0.362

std over 5 seeds
                AP   P@50  R@100
features                        
Full         0.021  0.036  0.018
No-shortcut  0.020  0.036  0.011

largest coefficients:
                         coef (per 1 SD)  odds ratio
system_sponsor_share              -0.477       0.621
high_band_share                   -0.448       0.639
n_engagements                     -0.447       0.639
link_site_of                      -0.394       0.674
missing:amount_cv                 -0.345       0.708
expert_hcp_consultant              0.202       1.224
svc_flag_gov                       0.204       1.2

## Cell 12 · Model 3：Decision Tree（决策树）

### 为什么可以用，以及它改进了什么

前两个模型都假设每个变量的作用是一条直线。但合规风险更像**规则**：

- **阈值**：`under_level_approvals` 只要 ≥ 1 次就很可疑，1 次和 5 次差别不大，不是直线关系。
- **交互**：某个变量只在另一个条件成立时才有意义，比如"提供中介类服务"本身很正常，但"中介类服务 + 大量敏感费用"就不一样。

决策树反复问"变量 ≤ 某个值吗？"，把供应商一层层分组，每个叶子节点里正例的比例就是风险分数。它**自动找阈值、自动表示交互**，而且：

- 不需要 log 变换、截尾、标准化（只看排序，Cell 6 说明过）；
- **最容易给分析员解释**：每家供应商的分数对应一条清晰的"如果……并且……则……"路径。

### 怎么设

| 项目 | 设置 | 理由 |
|---|---|---|
| 输入 | 原始 `X`（不做 log/截尾/标准化），只做 T3 填补 | 树对单调变换不敏感 |
| 最大深度 | `max_depth = 4` | 最多 4 层规则，人能读懂；太深会把练习集的 159 个正例背下来（过拟合） |
| 叶子最少样本 | `min_samples_leaf = 15` | 每条规则至少覆盖 15 家供应商，避免"为一两家专门造一条规则" |
| 类别加权 | `class_weight="balanced"` | 同 T5 |
| 输出 | 所在叶子的正例比例（加权后） | |

### 预期的限制

单棵树**不稳定**：数据稍微变一点，第一层选的变量就可能不同，整棵树都会变。分数也只有十几个不同取值（每个叶子一个），排名里并列很多。

In [6]:
# ============================================================
# Cell 13. model_tree
# ============================================================
from sklearn.tree import DecisionTreeClassifier, export_text

model_tree = make_pipeline(*tree_prep(),
                           DecisionTreeClassifier(max_depth=4, min_samples_leaf=15,
                                                  class_weight="balanced", random_state=0))
mean_tree, std_tree = run("3 Decision tree", model_tree, X)
print("mean over 5 seeds"); print(mean_tree.to_string())
print("\nstd over 5 seeds"); print(std_tree[["AP", "P@50", "R@100"]].to_string())

# ---- 树的规则（No-shortcut，全部数据拟合，仅用于解释）----
cols = FEATURE_SETS["No-shortcut"]
fit_tree = clone(model_tree).fit(X[cols], y)
names = list(cols) + [f"missing:{c}" for c in np.array(cols)[fit_tree[0].indicator_.features_]]
print("\ntree rules (class 1 = belongs to a labelled case):")
print(export_text(fit_tree[-1], feature_names=names, show_weights=False, decimals=2))
print(f"distinct score values: {len(np.unique(fit_tree.predict_proba(X[cols])[:, 1]))}")

mean over 5 seeds
                AP  P@10   P@25  P@50  P@100  R@100  first_hit  case_hit@100  member_recall@100
features                                                                                       
Full         0.419  0.72  0.592   0.6  0.562  0.353        1.0         0.558              0.374
No-shortcut  0.467  0.96  0.968   0.9  0.558  0.351        1.0         0.485              0.315

std over 5 seeds
                AP   P@50  R@100
features                        
Full         0.011  0.040  0.023
No-shortcut  0.008  0.037  0.023

tree rules (class 1 = belongs to a labelled case):
|--- under_level_approvals <= 0.50
|   |--- sensitive_expense_share <= 0.79
|   |   |--- log_total_usd <= 14.77
|   |   |   |--- activity_density <= 0.09
|   |   |   |   |--- class: 0
|   |   |   |--- activity_density >  0.09
|   |   |   |   |--- class: 0
|   |   |--- log_total_usd >  14.77
|   |   |   |--- active_months <= 51.50
|   |   |   |   |--- class: 0
|   |   |   |--- active_months >  

## Cell 14 · Model 4：Random Forest（随机森林）

### 为什么可以用，以及它改进了什么

针对单棵树**不稳定**的问题：训练很多棵树，每棵树

1. 用**有放回抽样**得到的不同样本训练（bootstrap，自助抽样）；
2. 每次分裂只从**随机挑选的一部分变量**里选（`max_features="sqrt"`，约 $\sqrt{67} \approx 8$ 个）。

最后把所有树的预测取平均。每棵树各有各的偏差，平均之后方差大幅下降，结果更稳定；同时保留了树能表示阈值和交互的优点。分数也变成连续的，排名不再大量并列。

### 怎么设

| 项目 | 设置 | 理由 |
|---|---|---|
| 输入 | 原始 `X`，只做 T3 填补 | 同决策树 |
| 树的数量 | `n_estimators = 500` | 足够多，结果不再随树的数量明显变化 |
| 叶子最少样本 | `min_samples_leaf = 3` | 单棵树可以长得比 Model 3 更深，过拟合靠"平均"来抵消 |
| 每次分裂候选变量 | `max_features = "sqrt"` | 让树彼此不同，平均才有效 |
| 类别加权 | `class_weight = "balanced_subsample"` | 每棵树按自己的抽样重新计算权重 |

### 代价

**可解释性下降**：500 棵树没法逐条读规则，只能看变量重要性（feature importance）。这正是 dossier 关心的权衡：*"the point at which additional model complexity stops earning its cost"*，即更复杂的模型带来的提升，是否值得失去的可解释性。

In [7]:
# ============================================================
# Cell 15. model_rf
# ============================================================
from sklearn.ensemble import RandomForestClassifier

model_rf = make_pipeline(*tree_prep(),
                         RandomForestClassifier(n_estimators=500, min_samples_leaf=3,
                                                max_features="sqrt",
                                                class_weight="balanced_subsample",
                                                n_jobs=-1, random_state=0))
mean_rf, std_rf = run("4 Random forest", model_rf, X)
print("mean over 5 seeds"); print(mean_rf.to_string())
print("\nstd over 5 seeds"); print(std_rf[["AP", "P@50", "R@100"]].to_string())

# ---- 变量重要性（No-shortcut，全部数据拟合）----
# 注意：基于不纯度（impurity）的重要性偏向取值多的连续变量，只作参考
cols = FEATURE_SETS["No-shortcut"]
fit_rf = clone(model_rf).fit(X[cols], y)
names = list(cols) + [f"missing:{c}" for c in np.array(cols)[fit_rf[0].indicator_.features_]]
imp_rf = pd.Series(fit_rf[-1].feature_importances_, index=names).sort_values(ascending=False)
print("\ntop 12 feature importances:"); print(imp_rf.head(12).round(3).to_string())

mean over 5 seeds
                AP  P@10   P@25   P@50  P@100  R@100  first_hit  case_hit@100  member_recall@100
features                                                                                        
Full         0.578  0.76  0.704  0.704  0.684  0.430        1.2         0.691              0.484
No-shortcut  0.510  0.88  0.936  0.904  0.600  0.377        1.4         0.515              0.373

std over 5 seeds
                AP   P@50  R@100
features                        
Full         0.014  0.052  0.020
No-shortcut  0.016  0.017  0.013



top 12 feature importances:
under_level_approvals    0.079
log_total_usd            0.077
activity_density         0.071
amount_cv                0.051
non_po_invoice_share     0.050
active_months            0.049
round_amount_share       0.047
approval_lead_days       0.044
invoices_per_payment     0.042
dd_lead_days             0.039
eng_record_lag_days      0.034
manual_payment_share     0.032


## Cell 16 · Model 5：Isolation Forest（孤立森林，无监督）

### 为什么需要它

Model 1–4 都需要 $y$。但：

- 主数据集约 2 万条记录，**没有标签**；
- 练习集只有 33 个 case，而且 dossier 明确说主数据集**包含练习集里没有的模式**，在练习集上学到的规律不一定能迁移。

所以需要一个**完全不看 $y$** 的模型：只问"哪些供应商和大多数供应商不一样"。$y$ 只用来**事后评估**它找到的异常和标记 case 是否重合。

### 原理

随机选一个变量、在它的范围内随机选一个切分点，把数据一分为二，重复下去，直到每个点都被单独隔开。**越异常的点越容易被隔开**，需要的切分次数越少。对很多棵随机树取平均路径长度，路径越短，异常分数越高。

### 怎么设

| 项目 | 设置 | 理由 |
|---|---|---|
| 输入 | `X_lin`，做 T3 填补和 T4 标准化，不截尾 | 截尾会把极端值拉回来，正好抹掉它要找的异常 |
| 树的数量 | `n_estimators = 500` | |
| 训练 | 全部数据，**不用 $y$，不需要交叉验证** | 无监督，没有"见过答案"的问题 |
| 输出 | 异常分数（越大越异常） | dossier 提醒：这是**排名**，不是概率 |

### 预期

"异常"不等于"有风险"。很多真实 case 的成员单看自己的行很普通（问题出在它们之间的关系上），所以孤立森林在 Layer 1 预计明显弱于有监督模型。但它是主数据集上**唯一不依赖练习标签**的方法，到 Layer 2、3 加入关系变量后可能会提升。

In [8]:
# ============================================================
# Cell 17. model_iforest
# ============================================================
from sklearn.ensemble import IsolationForest

model_iforest = make_pipeline(SimpleImputer(strategy="median", add_indicator=True),
                              StandardScaler(),
                              IsolationForest(n_estimators=500, random_state=0))
mean_if, std_if = run("5 Isolation forest", model_iforest, X_lin, supervised=False)
print("mean over 5 seeds"); print(mean_if.to_string())
print("\nstd over 5 seeds"); print(std_if[["AP", "P@50", "R@100"]].to_string())

mean over 5 seeds
                AP  P@10   P@25   P@50  P@100  R@100  first_hit  case_hit@100  member_recall@100
features                                                                                        
Full         0.195   0.0  0.064  0.200  0.246  0.155       21.0         0.509              0.247
No-shortcut  0.182   0.0  0.096  0.184  0.214  0.135       20.4         0.485              0.243

std over 5 seeds
                AP   P@50  R@100
features                        
Full         0.004  0.032  0.012
No-shortcut  0.004  0.017  0.007


## Cell 18 · Model 6：Multi-level Scorecard（多档计分卡，`model_ms`）

### 这是什么

这是我们最早设想的**传统打分方法**：不让模型自由学一条公式，而是按固定规则给每个变量分档打分，再加权求和。

$$
\text{总分}_i \;=\; \sum_{j=1}^{p} w_j \cdot \text{子分}_{ij},
\qquad \text{子分}_{ij} \in [0, 10],
\qquad \sum_j w_j = 10
\;\;\Rightarrow\;\; \text{总分}_i \in [0, 100]
$$

分三步：

1. **分档**：每个变量按切点（threshold）分成几档；
2. **档内分值**：每一档对应一个 0–10 的子分；
3. **跨变量权重**：每个变量一个权重 $w_j$。

和 Model 1–4 的区别：变量不是以原始数值进入公式，而是先变成"第几档"；结构固定，分析员能直接读懂"这家供应商在审批违规这一项拿了 10 分，在付款活跃度上拿了 7.5 分……"。

### 每个变量分几档（规则）

分档数取决于变量本身能支持几档，不能统一定成 4 或 5 档：

| 变量类型 | 判断方法 | 档数 | 切点怎么定 |
|---|---|---|---|
| **二元** | 只有两个取值 | 2 | 两个取值之间 |
| **集中在一个值** | 最常见的值占 ≥ 60%（比如 96% 的供应商 `under_level_approvals` = 0） | 3 | ① 最常见值；② 高于它的那些值的中位数。即"常见值 / 略高于常见值 / 明显高于常见值" |
| **连续** | 其余 | 5 | 五分位数（20%、40%、60%、80%）。如果有重复值导致切点重合，自动合并，档数可能少于 5 |

- **方向**：有些变量是越小越可疑（比如合同数）。用训练数据中"档位与 $y$ 的秩相关"的符号确定方向，保证 10 分永远给最可疑的那一档。
- **缺失值**：先用训练数据的中位数填补，再分档（同 Cell 6 的 T3）。
- **不需要 log 和标准化**：分档只看数值顺序，单调变换不影响分到哪一档。
- **防止泄漏**：交叉验证时，切点、方向、档内分值、权重都**只用训练折**计算。下面展示的是用全部 1,002 家计算的版本，仅用于说明。

### 每档的子分：两种做法

| 做法 | 子分 | 说明 |
|---|---|---|
| **等距**（`points="equal"`） | 第 $k$ 档（共 $K$ 档，$k = 0, \dots, K-1$）得 $10 \cdot k/(K-1)$：2 档 0/10，3 档 0/5/10，5 档 0/2.5/5/7.5/10 | 我们最初的设想。最低档给 0 分（"没有风险迹象"），和原方案 2/4/6/8/10 相比，每个变量只差一个对所有人相同的常数，**排名完全一样** |
| **按正例比例**（`points="rate"`） | 每档的正例比例（加平滑，避免小档极端）线性缩放到 0–10：比例最低的档 0 分，最高的档 10 分 | 不假设"风险随档位均匀上升"，比如 0 → 1 次是质变、1 → 5 次差别不大，也能表示 |

下面的 Cell 19 列出全部变量的档数和切点。

In [9]:
# ============================================================
# Cell 19. Model 6 结构：每个变量分几档、切点（threshold）在哪
# ============================================================
from sklearn.base import ClassifierMixin
from scipy.optimize import minimize
from scipy.stats import spearmanr

TYPE_ZH = {"binary": "二元", "concentrated": "集中在一个值", "continuous": "连续"}

def fit_nonneg_logistic(S, yv, C=1.0):
    """权重 >= 0 的逻辑回归（L2 正则 + 类别平衡加权）。S: 子分矩阵 (0-10)。"""
    Z = S / 10.0
    n, J = Z.shape
    npos = yv.sum()
    sw = np.where(yv == 1, n / (2 * npos), n / (2 * (n - npos)))
    def f(theta):
        b0, w = theta[0], theta[1:]
        z = b0 + Z @ w
        loss = C * np.sum(sw * (np.logaddexp(0, z) - yv * z)) + 0.5 * w @ w
        r = sw * (1 / (1 + np.exp(-z)) - yv)
        return loss, np.r_[C * r.sum(), C * Z.T @ r + w]
    res = minimize(f, np.zeros(J + 1), jac=True, method="L-BFGS-B",
                   bounds=[(None, None)] + [(0, None)] * J)
    return res.x[1:], res.x[0]

class MultiLevelScorecard(BaseEstimator, ClassifierMixin):
    """
    多档计分卡：
      1. 每个变量按类型分档（二元 2 档；集中在一个值 3 档；连续 5 档，按分位数）
      2. 每档给 0-10 的子分：points="equal" 等距；points="rate" 按该档正例比例
      3. 总分 = sum_j w_j * 子分_j，sum_j w_j = 10，总分 0-100
         weights="equal" 每个变量同权；weights="learned" 非负逻辑回归学出来
    所有切点、方向、子分、权重都只用 fit 时给的数据（交叉验证时 = 训练折）。
    """
    def __init__(self, points="equal", weights="learned", C=1.0,
                 concentrated_share=0.6, n_quantile_bins=5, smoothing=5.0):
        self.points, self.weights, self.C = points, weights, C
        self.concentrated_share, self.n_quantile_bins = concentrated_share, n_quantile_bins
        self.smoothing = smoothing

    def _cuts(self, s):
        u = np.unique(s)
        if len(u) <= 2:
            return "binary", [float(u[0])] if len(u) == 2 else []
        vals, cnt = np.unique(s, return_counts=True)
        mode, share = vals[cnt.argmax()], cnt.max() / len(s)
        if share >= self.concentrated_share:
            above = s[s > mode]
            if len(np.unique(above)) <= 1:
                return "concentrated", [float(mode)]
            q = np.quantile(above, 0.5)
            if q >= above.max():
                q = np.unique(above)[-2]
            return "concentrated", [float(mode), float(q)]
        qs = np.quantile(s, np.linspace(0, 1, self.n_quantile_bins + 1)[1:-1])
        cuts = sorted(set(float(q) for q in qs if q < s.max()))
        return "continuous", cuts

    def fit(self, Xd, yd):
        Xd = pd.DataFrame(Xd)
        yv = np.asarray(yd).astype(float)
        self.columns_ = list(Xd.columns)
        self.median_ = Xd.median()
        Xf = Xd.fillna(self.median_)
        p = yv.mean()
        self.bins_ = {}
        for c in self.columns_:
            s = Xf[c].to_numpy(float)
            kind, cuts = self._cuts(s)
            b = np.searchsorted(cuts, s, side="left")
            K = len(cuts) + 1
            n = np.bincount(b, minlength=K)
            pos = np.bincount(b, weights=yv, minlength=K)
            rate = (pos + self.smoothing * p) / (n + self.smoothing)    # 平滑后的正例比例
            rho = spearmanr(b, yv).correlation if K > 1 else 0.0
            ascending = not (rho < 0)                                      # nan 视为正向
            if K == 1:
                pts = np.zeros(1)
            elif self.points == "equal":
                order = np.arange(K) if ascending else np.arange(K)[::-1]
                pts = 10.0 * order / (K - 1)
            else:
                rng_ = rate.max() - rate.min()
                pts = 10.0 * (rate - rate.min()) / rng_ if rng_ > 0 else np.zeros(K)
            self.bins_[c] = dict(kind=kind, cuts=cuts, points=pts, n=n, rate=pos / np.maximum(n, 1),
                                 ascending=ascending)
        S = self.subscores(Xd).to_numpy()
        if self.weights == "equal":
            w = np.ones(S.shape[1])
        else:
            w, _ = fit_nonneg_logistic(S, yv, self.C)
        self.raw_weights_ = pd.Series(w, index=self.columns_)
        total = w.sum()
        self.weights_ = (self.raw_weights_ * 10 / total if total > 0
                         else pd.Series(10 / len(w), index=self.columns_))
        self.classes_ = np.array([0, 1])
        return self

    def subscores(self, Xd):
        Xf = pd.DataFrame(Xd)[self.columns_].fillna(self.median_)
        return pd.DataFrame({c: self.bins_[c]["points"][np.searchsorted(self.bins_[c]["cuts"],
                                                                        Xf[c].to_numpy(float), side="left")]
                             for c in self.columns_}, index=Xf.index)

    def total_score(self, Xd):
        return self.subscores(Xd) @ self.weights_          # 0-100

    def predict_proba(self, Xd):
        t = self.total_score(Xd).to_numpy() / 100.0
        return np.c_[1 - t, t]

def bin_labels(cuts):
    fmt = lambda v: f"{v:.3g}"
    if not cuts:
        return ["全部"]
    labs = [f"≤ {fmt(cuts[0])}"]
    labs += [f"({fmt(a)}, {fmt(b)}]" for a, b in zip(cuts[:-1], cuts[1:])]
    labs += [f"> {fmt(cuts[-1])}"]
    return labs

def structure_table(sc):
    rows = []
    for c in sc.columns_:
        b = sc.bins_[c]
        rows.append({"variable": c, "type": TYPE_ZH[b["kind"]], "n_bins": len(b["cuts"]) + 1,
                     "thresholds (切点)": ", ".join(f"{v:.3g}" for v in b["cuts"]),
                     "direction": "越大越可疑" if b["ascending"] else "越小越可疑"})
    return pd.DataFrame(rows).set_index("variable")


cols_ns = FEATURE_SETS["No-shortcut"]
sc_struct = MultiLevelScorecard(points="equal", weights="equal").fit(X[cols_ns], y)
struct = structure_table(sc_struct)
print(f"{len(struct)} variables (No-shortcut set)")
print(struct["type"].value_counts().to_string())
print(struct["n_bins"].value_counts().sort_index().rename("n_variables").to_string())
struct

60 variables (No-shortcut set)
type
二元        26
集中在一个值    18
连续        16
n_bins
2    27
3    19
4     2
5    12


,type,n_bins,thresholds (切点),direction
variable,,,,
is_individual,二元,2,0,越小越可疑
expert_hcp_consultant,二元,2,0,越大越可疑
expert_investigator_kol,二元,2,0,越小越可疑
expert_regulatory_access,二元,2,0,越小越可疑
expert_other,二元,2,0,越小越可疑
has_lei,二元,2,0,越小越可疑
has_registry_no,二元,2,0,越大越可疑
remit_country_mismatch,二元,2,0,越大越可疑
migrated_record,二元,2,0,越小越可疑


## Cell 20 · 每个变量分几档、每档多少分

下表用全部 1,002 家供应商计算（交叉验证时每个训练折会重新计算，切点会略有不同）。

**读法**：每一档写成 `范围 → 等距子分 / 按正例比例子分（这一档有几家，其中正例占多少）`。例如 `under_level_approvals`（审批人级别不够的次数）一行：`≤ 0` 得 **0** 分（等距）或 0.0 分（按比例），这一档 964 家、正例 13%；`(0, 1]` 得 **5** 分（等距）或 10.0 分（按比例），这一档 23 家、正例 100%；`> 1` 得 **10** 分（等距）或 9.2 分（按比例），这一档 15 家、正例 100%。注意等距版本里只违规 1 次的供应商只拿一半分数，虽然这一档全部是正例；按比例的版本没有这个问题。

从"正例比例"一栏可以直接看出这个变量有没有区分力：如果各档正例比例都接近整体的 16%，这个变量不管给多少分都没有用。

| 变量 | 含义 | 类型 | 档数 | 方向 | 各档：范围 → 等距子分 / 按比例子分（家数，正例比例） |
|---|---|---|---|---|---|
| `is_individual` | 是否个人专家 | 二元 | 2 | 越小越可疑 | ≤ 0 → **10** / 10.0（863 家，正例 17%）<br>> 0 → **0** / 0.0（139 家，正例 9%） |
| `expert_hcp_consultant` | 专家类型：医生顾问 | 二元 | 2 | 越大越可疑 | ≤ 0 → **0** / 0.0（934 家，正例 16%）<br>> 0 → **10** / 10.0（68 家，正例 18%） |
| `expert_investigator_kol` | 专家类型：临床研究者/KOL | 二元 | 2 | 越小越可疑 | ≤ 0 → **10** / 10.0（967 家，正例 16%）<br>> 0 → **0** / 0.0（35 家，正例 3%） |
| `expert_regulatory_access` | 专家类型：注册/准入专家 | 二元 | 2 | 越小越可疑 | ≤ 0 → **10** / 10.0（990 家，正例 16%）<br>> 0 → **0** / 0.0（12 家，正例 0%） |
| `expert_other` | 专家类型：其他（合并 3 个稀少类别） | 二元 | 2 | 越小越可疑 | ≤ 0 → **10** / 10.0（978 家，正例 16%）<br>> 0 → **0** / 0.0（24 家，正例 0%） |
| `has_lei` | 是否有 LEI 法人识别码 | 二元 | 2 | 越小越可疑 | ≤ 0 → **10** / 10.0（674 家，正例 16%）<br>> 0 → **0** / 0.0（328 家，正例 15%） |
| `has_registry_no` | 是否有公司注册号 | 二元 | 2 | 越大越可疑 | ≤ 0 → **0** / 0.0（638 家，正例 15%）<br>> 0 → **10** / 10.0（364 家，正例 17%） |
| `remit_country_mismatch` | 收款国 ≠ 所在国 | 二元 | 2 | 越大越可疑 | ≤ 0 → **0** / 0.0（974 家，正例 16%）<br>> 0 → **10** / 10.0（28 家，正例 25%） |
| `migrated_record` | 是否 ERP 迁移记录 | 二元 | 2 | 越小越可疑 | ≤ 0 → **10** / 10.0（833 家，正例 16%）<br>> 0 → **0** / 0.0（169 家，正例 14%） |
| `onboarded_in_event_window` | 是否在收购整合/疫苗上市期间开始合作 | 二元 | 2 | 越小越可疑 | ≤ 0 → **10** / 10.0（868 家，正例 16%）<br>> 0 → **0** / 0.0（134 家，正例 13%） |
| `has_declared_group` | 是否自报所属集团 | 二元 | 2 | 越大越可疑 | ≤ 0 → **0** / 0.0（914 家，正例 16%）<br>> 0 → **10** / 10.0（88 家，正例 19%） |
| `public_email` | 是否使用公共邮箱 | 二元 | 2 | 越小越可疑 | ≤ 0 → **10** / 10.0（893 家，正例 16%）<br>> 0 → **0** / 0.0（109 家，正例 11%） |
| `q_public_official_interaction` | 是否代表公司和政府官员打交道 | 二元 | 2 | 越大越可疑 | ≤ 0 → **0** / 0.0（874 家，正例 14%）<br>> 0 → **10** / 10.0（128 家，正例 28%） |
| `dd_sensitive_answer` | 自报国有/PEP相关/前政府官员（任一） | 二元 | 2 | 越小越可疑 | ≤ 0 → **10** / 10.0（990 家，正例 16%）<br>> 0 → **0** / 0.0（12 家，正例 8%） |
| `adverse_media` | 负面新闻 | 二元 | 2 | 越大越可疑 | ≤ 0 → **0** / 0.0（980 家，正例 16%）<br>> 0 → **10** / 10.0（22 家，正例 23%） |
| `screening_potential_match` | 筛查疑似命中（已排除） | 二元 | 2 | 越小越可疑 | ≤ 0 → **10** / 10.0（962 家，正例 16%）<br>> 0 → **0** / 0.0（40 家，正例 12%） |
| `dd_enhanced` | 是否做了加强尽调 | 二元 | 2 | 越大越可疑 | ≤ 0 → **0** / 0.0（855 家，正例 14%）<br>> 0 → **10** / 10.0（147 家，正例 27%） |
| `dd_lead_days` | 尽调完成到开始合作的天数 | 连续 | 5 | 越大越可疑 | ≤ 16 → **0** / 5.0（220 家，正例 16%）<br>(16, 27] → **2.5** / 0.0（191 家，正例 14%）<br>(27, 39] → **5** / 6.2（202 家，正例 16%）<br>(39, 51] → **7.5** / 10.0（198 家，正例 18%）<br>> 51 → **10** / 3.0（191 家，正例 15%） |
| `n_service_parents` | 提供几大类服务 | 集中在一个值 | 3 | 越小越可疑 | ≤ 1 → **10** / 3.7（880 家，正例 16%）<br>(1, 2] → **5** / 0.0（120 家，正例 11%）<br>> 2 → **0** / 10.0（2 家，正例 50%） |
| `svc_flag_hcp` | 提供接触医生的服务 | 二元 | 2 | 越小越可疑 | ≤ 0 → **10** / 10.0（475 家，正例 16%）<br>> 0 → **0** / 0.0（527 家，正例 16%） |
| `svc_flag_gov` | 提供接触政府的服务 | 二元 | 2 | 越大越可疑 | ≤ 0 → **0** / 0.0（652 家，正例 14%）<br>> 0 → **10** / 10.0（350 家，正例 19%） |
| `svc_flag_int` | 提供中介类服务 | 二元 | 2 | 越大越可疑 | ≤ 0 → **0** / 0.0（826 家，正例 15%）<br>> 0 → **10** / 10.0（176 家，正例 22%） |
| `svc_flag_flow` | 提供资金/货物流量大的服务 | 二元 | 2 | 越大越可疑 | ≤ 0 → **0** / 0.0（724 家，正例 15%）<br>> 0 → **10** / 10.0（278 家，正例 19%） |
| `n_engagements` | 合同数 | 连续 | 5 | 越小越可疑 | ≤ 1 → **10** / 9.9（282 家，正例 18%）<br>(1, 2] → **7.5** / 5.2（285 家，正例 15%）<br>(2, 3] → **5** / 0.0（190 家，正例 12%）<br>(3, 4] → **2.5** / 10.0（129 家，正例 18%）<br>> 4 → **0** / 9.0（116 家，正例 17%） |
| `n_eng_markets` | 合同涉及几个市场 | 集中在一个值 | 3 | 越大越可疑 | ≤ 1 → **0** / 0.0（909 家，正例 16%）<br>(1, 2] → **5** / 1.1（88 家，正例 17%）<br>> 2 → **10** / 10.0（5 家，正例 40%） |
| `n_business_units` | 涉及几个业务部门 | 连续 | 3 | 越大越可疑 | ≤ 1 → **0** / 0.0（612 家，正例 12%）<br>(1, 2] → **5** / 5.3（295 家，正例 20%）<br>> 2 → **10** / 10.0（95 家，正例 28%） |
| `n_contract_types` | 几种合同类型 | 集中在一个值 | 3 | 越小越可疑 | ≤ 1 → **10** / 1.3（951 家，正例 16%）<br>(1, 2] → **5** / 0.0（49 家，正例 14%）<br>> 2 → **0** / 10.0（2 家，正例 50%） |
| `high_band_share` | 高金额档位(3-4)合同比例 | 连续 | 4 | 越小越可疑 | ≤ 0 → **10** / 3.8（590 家，正例 16%）<br>(0, 0.2] → **6.66667** / 5.4（12 家，正例 17%）<br>(0.2, 0.8] → **3.33333** / 10.0（200 家，正例 19%）<br>> 0.8 → **0** / 0.0（200 家，正例 14%） |
| `missing_sponsor_share` | 缺发起人的合同比例 | 集中在一个值 | 3 | 越小越可疑 | ≤ 0 → **10** / 10.0（877 家，正例 16%）<br>(0, 0.5] → **5** / 8.9（93 家，正例 15%）<br>> 0.5 → **0** / 0.0（32 家，正例 3%） |
| `system_sponsor_share` | 发起人为系统默认账号的比例 | 集中在一个值 | 3 | 越小越可疑 | ≤ 0 → **10** / 10.0（962 家，正例 16%）<br>(0, 0.333] → **5** / 1.0（27 家，正例 4%）<br>> 0.333 → **0** / 0.0（13 家，正例 0%） |
| `open_engagements` | t 时仍在执行的合同数 | 集中在一个值 | 3 | 越大越可疑 | ≤ 0 → **0** / 0.0（728 家，正例 13%）<br>(0, 2] → **5** / 8.2（169 家，正例 22%）<br>> 2 → **10** / 10.0（105 家，正例 25%） |
| `eng_record_lag_days` | 合同开始到录入的平均天数 | 连续 | 5 | 越大越可疑 | ≤ 1.25 → **0** / 7.3（202 家，正例 17%）<br>(1.25, 2] → **2.5** / 0.0（219 家，正例 13%）<br>(2, 3] → **5** / 10.0（229 家，正例 18%）<br>(3, 4.5] → **7.5** / 1.5（162 家，正例 14%）<br>> 4.5 → **10** / 8.2（190 家，正例 17%） |
| `missing_approver_share` | 缺审批人的财务审批比例 | 集中在一个值 | 3 | 越小越可疑 | ≤ 0 → **10** / 10.0（754 家，正例 17%）<br>(0, 0.5] → **5** / 10.0（157 家，正例 17%）<br>> 0.5 → **0** / 0.0（91 家，正例 3%） |
| `under_level_approvals` | 审批人级别低于规定的次数 | 集中在一个值 | 3 | 越大越可疑 | ≤ 0 → **0** / 0.0（964 家，正例 13%）<br>(0, 1] → **5** / 10.0（23 家，正例 100%）<br>> 1 → **10** / 9.2（15 家，正例 100%） |
| `self_approvals` | 发起人自己审批的次数 | 集中在一个值 | 3 | 越大越可疑 | ≤ 0 → **0** / 6.7（952 家，正例 16%）<br>(0, 1] → **5** / 10.0（44 家，正例 20%）<br>> 1 → **10** / 0.0（6 家，正例 0%） |
| `missing_compliance_reviews` | 应做合规审查但没做的合同数 | 集中在一个值 | 3 | 越小越可疑 | ≤ 0 → **10** / 10.0（885 家，正例 17%）<br>(0, 2] → **5** / 4.3（87 家，正例 11%）<br>> 2 → **0** / 0.0（30 家，正例 7%） |
| `approval_lead_days` | 审批到合同开始的平均天数 | 连续 | 5 | 越大越可疑 | ≤ 12.5 → **0** / 7.3（208 家，正例 18%）<br>(12.5, 16] → **2.5** / 0.0（205 家，正例 11%）<br>(16, 18.5] → **5** / 5.1（192 家，正例 16%）<br>(18.5, 22] → **7.5** / 10.0（196 家，正例 20%）<br>> 22 → **10** / 4.3（201 家，正例 15%） |
| `log_total_usd` | 付款总额（对数） | 连续 | 5 | 越大越可疑 | ≤ 10.2 → **0** / 0.0（201 家，正例 9%）<br>(10.2, 12.1] → **2.5** / 7.2（200 家，正例 18%）<br>(12.1, 13.4] → **5** / 10.0（200 家，正例 22%）<br>(13.4, 14.9] → **7.5** / 10.0（200 家，正例 22%）<br>> 14.9 → **10** / 0.4（201 家，正例 9%） |
| `active_months` | 有付款的月数 | 连续 | 5 | 越小越可疑 | ≤ 7 → **10** / 1.2（226 家，正例 14%）<br>(7, 13] → **7.5** / 10.0（183 家，正例 23%）<br>(13, 20] → **5** / 0.0（193 家，正例 13%）<br>(20, 33] → **2.5** / 2.9（202 家，正例 16%）<br>> 33 → **0** / 1.2（198 家，正例 14%） |
| `activity_density` | 有付款月数 ÷ 合作月数 | 连续 | 5 | 越大越可疑 | ≤ 0.121 → **0** / 0.0（201 家，正例 7%）<br>(0.121, 0.25] → **2.5** / 5.4（200 家，正例 16%）<br>(0.25, 0.469] → **5** / 4.1（200 家，正例 14%）<br>(0.469, 0.89] → **7.5** / 6.9（200 家，正例 18%）<br>> 0.89 → **10** / 10.0（201 家，正例 23%） |
| `amount_cv` | 月付款额变异系数 | 连续 | 5 | 越大越可疑 | ≤ 0.362 → **0** / 0.6（201 家，正例 13%）<br>(0.362, 0.464] → **2.5** / 10.0（200 家，正例 20%）<br>(0.464, 0.578] → **5** / 1.3（200 家，正例 14%）<br>(0.578, 0.77] → **7.5** / 0.0（200 家，正例 13%）<br>> 0.77 → **10** / 7.2（201 家，正例 18%） |
| `round_amount_share` | 整数金额发票比例 | 连续 | 5 | 越大越可疑 | ≤ 0.0637 → **0** / 3.0（201 家，正例 15%）<br>(0.0637, 0.0719] → **2.5** / 9.4（200 家，正例 19%）<br>(0.0719, 0.0788] → **5** / 0.0（200 家，正例 13%）<br>(0.0788, 0.0885] → **7.5** / 0.0（200 家，正例 13%）<br>> 0.0885 → **10** / 10.0（201 家，正例 19%） |
| `manual_payment_share` | 手工付款比例 | 连续 | 5 | 越大越可疑 | ≤ 0 → **0** / 8.3（387 家，正例 17%）<br>(0, 0.00732] → **2.5** / 4.3（14 家，正例 14%）<br>(0.00732, 0.0189] → **5** / 0.0（201 家，正例 12%）<br>(0.0189, 0.0317] → **7.5** / 6.5（202 家，正例 16%）<br>> 0.0317 → **10** / 10.0（198 家，正例 18%） |
| `invoices_per_payment` | 每次付款对应发票数 | 连续 | 5 | 越大越可疑 | ≤ 1.04 → **0** / 7.1（201 家，正例 16%）<br>(1.04, 1.08] → **2.5** / 0.0（200 家，正例 14%）<br>(1.08, 1.1] → **5** / 2.9（200 家，正例 15%）<br>(1.1, 1.13] → **7.5** / 7.3（200 家，正例 16%）<br>> 1.13 → **10** / 10.0（201 家，正例 17%） |
| `non_po_invoice_share` | 无采购订单发票比例 | 连续 | 5 | 越小越可疑 | ≤ 0.115 → **10** / 10.0（201 家，正例 21%）<br>(0.115, 0.163] → **7.5** / 6.1（200 家，正例 17%）<br>(0.163, 0.2] → **5** / 2.6（229 家，正例 14%）<br>(0.2, 0.299] → **2.5** / 6.6（171 家，正例 18%）<br>> 0.299 → **0** / 0.0（201 家，正例 11%） |
| `p_card_share` | 公司采购卡付款比例 | 连续 | 4 | 越大越可疑 | ≤ 0 → **0** / 0.0（545 家，正例 15%）<br>(0, 0.02] → **3.33333** / 10.0（58 家，正例 19%）<br>(0.02, 0.0484] → **6.66667** / 0.5（199 家，正例 16%）<br>> 0.0484 → **10** / 3.2（200 家，正例 16%） |
| `expense_reimbursement_share` | 员工报销比例 | 集中在一个值 | 3 | 越小越可疑 | ≤ 0 → **10** / 10.0（919 家，正例 17%）<br>(0, 0.143] → **5** / 0.0（45 家，正例 4%）<br>> 0.143 → **0** / 2.9（38 家，正例 8%） |
| `n_expense_classes` | 费用类别数 | 集中在一个值 | 3 | 越大越可疑 | ≤ 1 → **0** / 0.0（942 家，正例 15%）<br>(1, 2] → **5** / 7.4（57 家，正例 21%）<br>> 2 → **10** / 10.0（3 家，正例 33%） |
| `tov_hcp_share` | 向医生/医疗机构利益输送类付款比例 | 连续 | 2 | 越大越可疑 | ≤ 0 → **0** / 0.0（545 家，正例 16%）<br>> 0 → **10** / 10.0（457 家，正例 16%） |
| `sensitive_expense_share` | 敏感费用比例（佣金/资助捐赠/赞助/政府费用） | 集中在一个值 | 3 | 越大越可疑 | ≤ 0 → **0** / 0.0（890 家，正例 13%）<br>(0, 0.915] → **5** / 3.5（12 家，正例 25%）<br>> 0.915 → **10** / 10.0（100 家，正例 41%） |
| `n_addresses` | 地址记录数 | 集中在一个值 | 3 | 越大越可疑 | ≤ 1 → **0** / 0.2（914 家，正例 16%）<br>(1, 2] → **5** / 0.0（86 家，正例 15%）<br>> 2 → **10** / 10.0（2 家，正例 100%） |
| `risky_address_type` | 用过注册代理/虚拟办公室/邮政信箱/共享办公 | 二元 | 2 | 越小越可疑 | ≤ 0 → **10** / 10.0（919 家，正例 16%）<br>> 0 → **0** / 0.0（83 家，正例 16%） |
| `org_at_residential` | 公司注册在住宅地址 | 二元 | 2 | 越小越可疑 | ≤ 0 → **10** / 10.0（975 家，正例 16%）<br>> 0 → **0** / 0.0（27 家，正例 15%） |
| `address_type_missing` | 地址类型未分类 | 二元 | 2 | 越大越可疑 | ≤ 0 → **0** / 0.0（598 家，正例 15%）<br>> 0 → **10** / 10.0（404 家，正例 17%） |
| `n_institution_links` | 关联机构数 | 集中在一个值 | 3 | 越大越可疑 | ≤ 0 → **0** / 1.7（829 家，正例 15%）<br>(0, 1] → **5** / 0.0（117 家，正例 9%）<br>> 1 → **10** / 10.0（56 家，正例 46%） |
| `public_institution_links` | 关联公立机构数 | 集中在一个值 | 3 | 越大越可疑 | ≤ 0 → **0** / 0.0（868 家，正例 15%）<br>(0, 1] → **5** / 0.4（106 家，正例 16%）<br>> 1 → **10** / 10.0（28 家，正例 50%） |
| `link_serves` | 作为分销商服务的机构数 | 集中在一个值 | 3 | 越大越可疑 | ≤ 0 → **0** / 0.0（947 家，正例 14%）<br>(0, 2] → **5** / 10.0（32 家，正例 47%）<br>> 2 → **10** / 8.5（23 家，正例 43%） |
| `link_site_of` | 作为临床试验点所属的机构数 | 集中在一个值 | 3 | 越小越可疑 | ≤ 0 → **10** / 5.0（952 家，正例 16%）<br>(0, 1] → **5** / 0.0（49 家，正例 2%）<br>> 1 → **0** / 10.0（1 家，正例 100%） |
| `link_affiliated_with` | 个人专家任职的机构数 | 二元 | 2 | 越小越可疑 | ≤ 0 → **10** / 10.0（934 家，正例 16%）<br>> 0 → **0** / 0.0（68 家，正例 13%） |
| `country_bo_access_low` | 所在国受益所有权登记公开程度低 | 二元 | 2 | 越大越可疑 | ≤ 0 → **0** / 0.0（365 家，正例 15%）<br>> 0 → **10** / 10.0（637 家，正例 16%） |

In [10]:
# ============================================================
# Cell 21. 权重的计算
#   (1) 相等权重：每个变量 w = 10 / 60
#   (2) 学习权重：非负逻辑回归（子分 -> y），再缩放到总和 10
#   并比较两种权重下，每个变量对排名的实际"影响力" = w_j × 子分_j 的标准差
# ============================================================
weight_tables = {}
for pts in ["equal", "rate"]:
    sc_e = MultiLevelScorecard(points=pts, weights="equal").fit(X[cols_ns], y)
    sc_l = MultiLevelScorecard(points=pts, weights="learned").fit(X[cols_ns], y)
    sd = sc_e.subscores(X[cols_ns]).std()
    t = pd.DataFrame({
        "n_bins": [len(sc_e.bins_[c]["cuts"]) + 1 for c in cols_ns],
        "subscore_sd": sd.values,
        "w_equal": sc_e.weights_.values,
        "w_learned": sc_l.weights_.values,
    }, index=cols_ns)
    t["influence_equal_%"] = 100 * t["w_equal"] * t["subscore_sd"] / (t["w_equal"] * t["subscore_sd"]).sum()
    t["influence_learned_%"] = 100 * t["w_learned"] * t["subscore_sd"] / (t["w_learned"] * t["subscore_sd"]).sum()
    weight_tables[pts] = t.sort_values("w_learned", ascending=False)

wt = weight_tables["equal"]
print("等距子分 + 两种权重（按学习权重排序，前 15 个）")
print(wt.head(15).round(3).to_string())
print(f"\n学习权重 = 0 的变量：{(wt['w_learned'] == 0).sum()} / {len(wt)}")
print("\n相等权重下，档数与影响力的关系（平均影响力 %）：")
print(wt.groupby("n_bins")[["influence_equal_%", "influence_learned_%"]].mean().round(2).to_string())
wt.round(3)

等距子分 + 两种权重（按学习权重排序，前 15 个）
                             n_bins  subscore_sd  w_equal  w_learned  influence_equal_%  influence_learned_%
under_level_approvals             3        1.415    0.167      1.969              0.808               11.775
system_sponsor_share              3        1.379    0.167      0.658              0.788                3.835
expert_hcp_consultant             2        2.516    0.167      0.524              1.437                5.569
n_business_units                  3        3.316    0.167      0.404              1.894                5.659
n_engagements                     5        3.323    0.167      0.391              1.898                5.482
expert_other                      2        1.530    0.167      0.388              0.874                2.509
activity_density                  5        3.541    0.167      0.363              2.022                5.437
sensitive_expense_share           3        3.028    0.167      0.360              1.730             

,n_bins,subscore_sd,w_equal,w_learned,influence_equal_%,influence_learned_%
under_level_approvals,3,1.415,0.167,1.969,0.808,11.775
system_sponsor_share,3,1.379,0.167,0.658,0.788,3.835
expert_hcp_consultant,2,2.516,0.167,0.524,1.437,5.569
n_business_units,3,3.316,0.167,0.404,1.894,5.659
n_engagements,5,3.323,0.167,0.391,1.898,5.482
expert_other,2,1.530,0.167,0.388,0.874,2.509
activity_density,5,3.541,0.167,0.363,2.022,5.437
sensitive_expense_share,3,3.028,0.167,0.360,1.730,4.599
high_band_share,4,4.201,0.167,0.356,2.399,6.316
expense_reimbursement_share,3,2.134,0.167,0.327,1.219,2.950


## Cell 22 · 权重是怎么得到的

### 1. 为什么不能直接给每个变量相同的权重

决定一个变量对**排名**影响多大的，不是它的满分（都是 10），而是**大家在这一项上的分数拉开了多少**，也就是子分的标准差。所以每个变量的实际影响力是

$$
\text{影响力}_j \;\propto\; w_j \times \text{sd}(\text{子分}_j)
$$

标准差取决于两件事：**分了几档**，以及**供应商在各档之间分布得均不均匀**。例如 5 档按五分位切，每档 20% 的人，分数拉得很开；而"集中在一个值"的变量虽然有 3 档，但 80%–96% 的人都在最低档，分数几乎拉不开。**相等权重下，影响力完全由这两件事决定，和变量重不重要无关。**

Cell 21 的实际结果（等距子分），按档数平均：

| 档数 | 相等权重下的平均影响力 | 学习权重下的平均影响力 |
|---|---|---|
| 2 档 | 1.81% | 1.32% |
| 3 档 | 1.15% | 2.14% |
| 4 档 | 2.38% | 3.40% |
| 5 档 | 2.03% | 1.40% |

（3 档的变量大多是"集中在一个值"的类型，所以相等权重下影响力反而最低。）

以两个变量为例：

- `under_level_approvals`（审批人级别不够），3 档，但 96% 的供应商在最低档，子分标准差只有 1.42，相等权重下影响力只有 **0.8%**。而它恰恰是最强的信号：违规过的 38 家全部是正例；
- `activity_density`（付款活跃度），5 档、各档人数均匀，子分标准差 3.54，相等权重下影响力 **2.0%**，是前者的 2 倍多，但区分力弱得多。

所以权重**必须调整**，而且调整的依据不应该是"分了几档"，而是"这个变量对区分正负例实际有多大帮助"。

### 2. 两种权重

**(a) 相等权重**：每个变量 $w_j = 10/60 \approx 0.167$。代表"所有变量同样重要"的纯人工设定，作为对照。

**(b) 学习权重**：让数据决定每个变量该占多少。把 60 个子分当作自变量，用**逻辑回归**拟合 $y$：

$$
\log \frac{P(y_i = 1)}{1 - P(y_i = 1)} \;=\; b_0 + \sum_j \beta_j \cdot \frac{\text{子分}_{ij}}{10}
$$

具体设置：

| 设置 | 选择 | 理由 |
|---|---|---|
| 系数约束 | $\beta_j \ge 0$ | 计分卡里"负权重"没有意义：子分越高本应越可疑。方向已经在分档时确定，权重只决定"有多重要" |
| 正则化 | L2，$C = 1$ | 60 个变量、159 个正例，防止系数过大、过于依赖练习集 |
| 类别加权 | 正例权重 ≈ 负例的 5.3 倍 | 同 Cell 6 的 T5 |
| 缩放 | $w_j = 10 \cdot \beta_j / \sum_k \beta_k$ | 让权重总和为 10，总分落在 0–100；等比例缩放不改变排名 |

学习权重会**自动校正分档带来的影响力差异**：一个变量如果档分得细、但没什么区分力，它的系数会自然变小，甚至为 0。

### 3. 学到的权重（等距子分，全部数据，前 8 个）

| 变量 | 含义 | 档数 | 相等权重 | 学习权重 | 影响力（相等） | 影响力（学习） |
|---|---|---|---|---|---|---|
| `under_level_approvals` | 审批人级别低于规定的次数 | 3 | 0.167 | **1.969** | 0.8% | 11.8% |
| `system_sponsor_share` | 发起人为系统默认账号的比例 | 3 | 0.167 | **0.658** | 0.8% | 3.8% |
| `expert_hcp_consultant` | 专家类型：医生顾问 | 2 | 0.167 | **0.524** | 1.4% | 5.6% |
| `n_business_units` | 涉及几个业务部门 | 3 | 0.167 | **0.404** | 1.9% | 5.7% |
| `n_engagements` | 合同数 | 5 | 0.167 | **0.391** | 1.9% | 5.5% |
| `expert_other` | 专家类型：其他（合并 3 个稀少类别） | 2 | 0.167 | **0.388** | 0.9% | 2.5% |
| `activity_density` | 有付款月数 ÷ 合作月数 | 5 | 0.167 | **0.363** | 2.0% | 5.4% |
| `sensitive_expense_share` | 敏感费用比例（佣金/资助捐赠/赞助/政府费用） | 3 | 0.167 | **0.360** | 1.7% | 4.6% |

- 60 个变量中有 **14** 个学习权重为 0：在其他变量已经考虑的情况下，它们没有额外贡献，相当于从计分卡里去掉。
- 交叉验证时，权重在每个训练折重新学习。

### 4. Model 6 的四个版本

| 版本 | 档内子分 | 跨变量权重 | 代表 |
|---|---|---|---|
| 6a | 等距 | 相等 | 纯人工规则 |
| **6b（`model_ms` 主版本）** | 等距 | 学习 | 我们最初的设计：等距分档，权重按数据调整 |
| 6c | 按正例比例 | 相等 | |
| 6d | 按正例比例 | 学习 | 标准的数据驱动计分卡 |

对比 6a 和 6b 可以回答"权重要不要调整"；对比 6b 和 6d 可以回答"等距分值是否合理"。

In [11]:
# ============================================================
# Cell 23. model_ms（Model 6）：能否从 1,002 家中把 159 家筛出来
# ============================================================
MS_VARIANTS = {
    "6a Scorecard equal-pts equal-wts": MultiLevelScorecard(points="equal", weights="equal"),
    "6b Scorecard equal-pts learned-wts": MultiLevelScorecard(points="equal", weights="learned"),
    "6c Scorecard rate-pts equal-wts": MultiLevelScorecard(points="rate", weights="equal"),
    "6d Scorecard rate-pts learned-wts": MultiLevelScorecard(points="rate", weights="learned"),
}
MS_NAME = "6b Scorecard equal-pts learned-wts"
model_ms = MS_VARIANTS[MS_NAME]

ms_summary = {}
for name, m in MS_VARIANTS.items():
    mean_, std_ = run(name, m, X)
    ms_summary[name] = mean_.loc["No-shortcut"]
ms_tbl = pd.DataFrame(ms_summary).T
print("No-shortcut, mean over 5 seeds")
print(ms_tbl[["AP", "P@10", "P@25", "P@50", "P@100", "R@100", "case_hit@100", "member_recall@100"]].to_string())

# ---- 前 k 名里能找到 159 家中的多少 ----
def capture(score, ks=(50, 100, 159, 200, 300, 500)):
    order = score.sort_values(ascending=False).index
    return {k: int(y.loc[order[:k]].sum()) for k in ks}

cap = pd.DataFrame({
    "Random (expected)": {k: round(k * y.mean(), 1) for k in (50, 100, 159, 200, 300, 500)},
    "2 Logistic regression": capture(SCORES[("2 Logistic regression", "No-shortcut")]),
    "4 Random forest": capture(SCORES[("4 Random forest", "No-shortcut")]),
    **{n: capture(SCORES[(n, "No-shortcut")]) for n in MS_VARIANTS},
}).T
cap.columns = [f"top {k}" for k in cap.columns]
print(f"\n前 k 名中找到的正例数（共 {y.sum()} 家；No-shortcut；5 个 seed 平均分数排序）")
print(cap.to_string())

# ---- 一家供应商的计分卡明细（主版本，全部数据拟合，仅用于展示）----
ms_fit = clone(model_ms).fit(X[cols_ns], y)
total = ms_fit.total_score(X[cols_ns]).sort_values(ascending=False)
example = total.index[0]
detail = pd.DataFrame({"value": X.loc[example, cols_ns], "subscore": ms_fit.subscores(X[cols_ns]).loc[example],
                       "weight": ms_fit.weights_})
detail["points"] = detail["subscore"] * detail["weight"]
print(f"\n计分卡明细：{example}，总分 {total.iloc[0]:.1f} / 100（y = {y[example]}）")
print(detail.sort_values("points", ascending=False).head(8).round(2).to_string())
cap

No-shortcut, mean over 5 seeds
                                       AP  P@10   P@25   P@50  P@100  R@100  case_hit@100  member_recall@100
6a Scorecard equal-pts equal-wts    0.220  0.32  0.368  0.372  0.290  0.182         0.509              0.308
6b Scorecard equal-pts learned-wts  0.416  1.00  0.896  0.708  0.482  0.303         0.594              0.387
6c Scorecard rate-pts equal-wts     0.250  0.32  0.432  0.412  0.306  0.192         0.533              0.307
6d Scorecard rate-pts learned-wts   0.398  0.92  0.824  0.652  0.448  0.282         0.430              0.258

前 k 名中找到的正例数（共 159 家；No-shortcut；5 个 seed 平均分数排序）
                                    top 50  top 100  top 159  top 200  top 300  top 500
Random (expected)                      7.9     15.9     25.2     31.7     47.6     79.3
2 Logistic regression                 41.0     54.0     63.0     66.0     80.0    112.0
4 Random forest                       47.0     61.0     72.0     75.0     92.0    119.0
6a Scorecard equal-pt

,top 50,top 100,top 159,top 200,top 300,top 500
Random (expected),7.9,15.9,25.2,31.7,47.6,79.3
2 Logistic regression,41.0,54.0,63.0,66.0,80.0,112.0
4 Random forest,47.0,61.0,72.0,75.0,92.0,119.0
6a Scorecard equal-pts equal-wts,21.0,35.0,46.0,54.0,63.0,87.0
6b Scorecard equal-pts learned-wts,39.0,54.0,64.0,70.0,80.0,101.0
6c Scorecard rate-pts equal-wts,23.0,37.0,43.0,57.0,75.0,104.0
6d Scorecard rate-pts learned-wts,38.0,51.0,62.0,68.0,89.0,105.0


## Cell 24 · 全部模型对比（Model 1 线性回归 → Model 6 计分卡）

读表时看四件事：

1. **每一步换模型带来了什么**：1 → 2 输出改成概率；2 → 3 允许阈值和交互；3 → 4 多棵树平均；5 不用标签；6 固定结构的计分卡。
2. **计分卡和统计模型差多少**：这是"传统规则打分"和"让模型学"的差距。
3. **Full vs No-shortcut**：差距越大，越依赖可能是练习集捷径的变量。
4. **标准差**：两个模型的差距小于标准差时，不能说一个比另一个好。

In [12]:
# ============================================================
# Cell 25. 对比表
# ============================================================
allres = pd.concat(RESULTS, ignore_index=True)
metrics = ["AP", "P@10", "P@25", "P@50", "P@100", "R@100", "first_hit",
           "case_hit@100", "member_recall@100"]
comp_mean = allres.groupby(["features", "model"], sort=False)[metrics].mean().round(3)
comp_std = allres.groupby(["features", "model"], sort=False)[["AP", "P@50"]].std().round(3)
comp = comp_mean.join(comp_std, rsuffix="_std")
print(f"random precision = {y.mean():.3f}\n")
for fs in FEATURE_SETS:
    print(f"===== {fs} =====")
    print(comp.loc[fs].to_string()); print()
comp

random precision = 0.159

===== Full =====
                                       AP  P@10   P@25   P@50  P@100  R@100  first_hit  case_hit@100  member_recall@100  AP_std  P@50_std
model                                                                                                                                    
Random                              0.166  0.12  0.168  0.168  0.164  0.103        5.6         0.345              0.095   0.009     0.018
1 Linear regression                 0.589  1.00  0.992  0.796  0.638  0.401        1.0         0.679              0.420   0.025     0.048
2 Logistic regression               0.602  1.00  0.976  0.808  0.660  0.415        1.0         0.691              0.412   0.021     0.036
3 Decision tree                     0.419  0.72  0.592  0.600  0.562  0.353        1.0         0.558              0.374   0.011     0.040
4 Random forest                     0.578  0.76  0.704  0.704  0.684  0.430        1.2         0.691              0.484   0.014  

,,AP,P@10,P@25,P@50,P@100,R@100,first_hit,case_hit@100,member_recall@100,AP_std,P@50_std
features,model,,,,,,,,,,,
Full,Random,0.166,0.12,0.168,0.168,0.164,0.103,5.6,0.345,0.095,0.009,0.018
No-shortcut,Random,0.166,0.12,0.168,0.168,0.164,0.103,5.6,0.345,0.095,0.009,0.018
Full,1 Linear regression,0.589,1.00,0.992,0.796,0.638,0.401,1.0,0.679,0.420,0.025,0.048
No-shortcut,1 Linear regression,0.487,1.00,1.000,0.892,0.556,0.350,1.0,0.527,0.386,0.020,0.030
Full,2 Logistic regression,0.602,1.00,0.976,0.808,0.660,0.415,1.0,0.691,0.412,0.021,0.036
No-shortcut,2 Logistic regression,0.462,1.00,1.000,0.812,0.504,0.317,1.0,0.533,0.362,0.020,0.036
Full,3 Decision tree,0.419,0.72,0.592,0.600,0.562,0.353,1.0,0.558,0.374,0.011,0.040
No-shortcut,3 Decision tree,0.467,0.96,0.968,0.900,0.558,0.351,1.0,0.485,0.315,0.008,0.037
Full,4 Random forest,0.578,0.76,0.704,0.704,0.684,0.430,1.2,0.691,0.484,0.014,0.052


## Cell 26 · 结论

数字为 5 个 seed 的平均；随机排序 AP ≈ 0.16。主要看 **No-shortcut**（去掉 7 个可能是练习集捷径的变量）。

### 1. 全部模型

| 模型 | AP（No-shortcut） | AP（Full） | 前 159 名找到的正例（共 159） |
|---|---|---|---|
| 随机排序 | 0.166 | 0.166 | 约 25 |
| 1 线性回归 | 0.487 | 0.589 | – |
| 2 逻辑回归 | 0.462 | 0.602 | 63 |
| 3 决策树 | 0.467 | 0.419 | – |
| 4 随机森林 | **0.510** | 0.578 | **72** |
| 5 孤立森林（无监督） | 0.182 | 0.195 | – |
| 6a 计分卡：等距子分 + 相等权重 | 0.220 | 0.292 | 46 |
| **6b 计分卡：等距子分 + 学习权重（`model_ms`）** | 0.416 | 0.509 | 64 |
| 6c 计分卡：按比例子分 + 相等权重 | 0.250 | 0.336 | 43 |
| 6d 计分卡：按比例子分 + 学习权重 | 0.398 | 0.559 | 62 |

### 2. 关于计分卡（Model 6）

1. **权重必须调整。** 相等权重的计分卡（6a、6c）AP 只有 0.22–0.25，只比随机好一点；换成学习权重后（6b、6d）升到 0.40–0.42，几乎翻倍。原因见 Cell 22：相等权重下，每个变量的影响力由"子分拉开了多少"决定，而不是由"它有多重要"决定。最强的信号 `under_level_approvals` 在相等权重下只占 0.8% 的影响力，学习权重后升到约 12%。
2. **等距分值基本可以接受。** 在 No-shortcut 上，等距（6b，0.416）和按正例比例（6d，0.398）的差距小于标准差（约 0.02–0.03）。在 Full 上按比例更好（0.559 vs 0.509）。等距的主要缺点是对"0 → 1 次就是质变"的变量处理不好，比如只违规 1 次的供应商只拿一半分数（Cell 20）。
3. **计分卡和逻辑回归在名单最前面几乎一样好。** 前 159 名中，6b 找到 64 家，逻辑回归 63 家；前 10 名都全对。整体 AP 低约 0.05，这是把连续数值切成档、丢掉档内差别的代价。换来的是**完全透明**：每家供应商的总分可以拆成"审批人级别不够这一项 19.7 分 + 其他各项……"（Cell 23 的明细），分析员可以直接核对。
4. **14 个变量学习权重为 0**，在其他变量已经考虑的情况下没有额外贡献，计分卡实际上只用了 46 个变量。

### 3. 关于 Layer 1 整体

1. **只看单行能找到一部分，但找不全。** 最好的随机森林在前 159 名中找到 72 家，约 45%；计分卡和逻辑回归约 40%；随机只有 16%。剩下一半以上的正例，单看自己的行和普通供应商没有区别。
2. **瓶颈是信息，不是模型。** 在 No-shortcut 上，有监督模型（Model 1–4、6b、6d）的 AP 都在 0.40–0.51 之间。从计分卡到随机森林，模型复杂了很多，提升只有约 0.1。
3. **无监督几乎找不到**（孤立森林 AP 0.18）：练习 case 的成员单看自己并不"异常"，问题出在它们之间的关系上。
4. **只能找到成员，不能找全 case**：前 100 名平均只覆盖每个 case 约 26%–39% 的成员。把成员连成完整的 case，需要 Layer 2 的关系信息。
5. **捷径变量影响很大**：多数模型从 Full 换到 No-shortcut，AP 下降 0.07–0.16。正式结论应以 No-shortcut 为准。

### 4. 后续各层沿用的模型

每一层**只加变量、不改模型设置**，提升就可以完全归因到新加的关系信息上。建议沿用三个：

- **逻辑回归**：阶梯比较的基准，可解释；
- **随机森林**：Layer 1 表现最好的有监督模型；
- **多档计分卡 6b（`model_ms`）**：代表传统的规则打分方法，用来回答项目的核心问题，即"关系信息能否发现传统打分方法发现不了的问题"。


In [13]:
# ============================================================
# Cell 27. 导出：Layer 1 基准模型分数 + reason codes
#
# 基准模型：Model 2 Logistic regression，No-shortcut 变量
#   - 后面每一层要用"同一个打分模型、只加变量"来比较，基准模型需要稳定、可解释；
#   - 逻辑回归的系数可以直接拆成每个变量对分数的贡献，用来写 reason codes。
# 同时导出所有模型（含计分卡）的分数，方便对照。
# ============================================================
BASE_MODEL, BASE_FS = "2 Logistic regression", "No-shortcut"
cols = FEATURE_SETS[BASE_FS]

Z = fit_logistic[:-1].transform(X_lin[cols])
names = list(cols) + [f"missing:{c}" for c in np.array(cols)[fit_logistic[0].indicator_.features_]]
contrib = pd.DataFrame(Z * fit_logistic[-1].coef_[0], index=X.index, columns=names)
desc = spec.set_index("variable")["meaning"]

def reasons(r, n=3):
    top = contrib.loc[r].sort_values(ascending=False).head(n)
    return "; ".join(f"{v} ({desc.get(v, v)}) = {X.at[r, v]:g}" if v in X.columns else v
                     for v in top.index if top[v] > 0)

ALL_MODELS = ["1 Linear regression", "2 Logistic regression", "3 Decision tree",
              "4 Random forest", "5 Isolation forest", *MS_VARIANTS]
out = pd.DataFrame({f"score_{m.split(' ', 1)[1].replace(' ', '_')}": SCORES[(m, BASE_FS)]
                    for m in ALL_MODELS})
out["risk_score"] = SCORES[(BASE_MODEL, BASE_FS)]
out["y"] = y
out = out.sort_values("risk_score", ascending=False)
out["rank"] = np.arange(1, len(out) + 1)
out["reason_codes"] = [reasons(r) for r in out.index]

submission = pd.DataFrame({
    "case_rank": out["rank"].to_numpy(),
    "member_record_ids": out.index,
    "typology_tags": "",                      # Layer 1 无法判断 typology
    "score": out["risk_score"].round(4).to_numpy(),
    "reason_codes": out["reason_codes"].to_numpy(),
})
Path("outputs").mkdir(exist_ok=True)
out.to_csv("outputs/layer1_scores.csv")
submission.to_csv("outputs/layer1_submission_format.csv", index=False)
X.assign(y=y).to_csv("outputs/layer1_features.csv")
allres.to_csv("outputs/layer1_model_results_by_seed.csv", index=False)
struct.to_csv("outputs/layer1_scorecard_bins.csv")
print("saved to outputs/: layer1_scores.csv, layer1_submission_format.csv, layer1_features.csv, "
      "layer1_model_results_by_seed.csv, layer1_scorecard_bins.csv")
out.head(10)

saved to outputs/: layer1_scores.csv, layer1_submission_format.csv, layer1_features.csv, layer1_model_results_by_seed.csv, layer1_scorecard_bins.csv


,score_Linear_regression,score_Logistic_regression,score_Decision_tree,score_Random_forest,score_Isolation_forest,score_Scorecard_equal-pts_equal-wts,score_Scorecard_equal-pts_learned-wts,score_Scorecard_rate-pts_equal-wts,score_Scorecard_rate-pts_learned-wts,risk_score,y,rank,reason_codes
record_id,,,,,,,,,,,,,
TP-E62F69EE,1.263696,0.999301,1.0,0.629952,0.481154,0.624167,0.730174,0.606487,0.619975,0.999301,1,1,under_level_approvals (审批人级别低于规定的次数) = 3; n_institution_links (关联机构数) = 2; n...
TP-FC036024,1.302219,0.999296,1.0,0.645533,0.422046,0.509167,0.736675,0.491958,0.613413,0.999296,1,2,under_level_approvals (审批人级别低于规定的次数) = 2; n_business_units (涉及几个业务部门) = 3; d...
TP-ABC566F2,1.225184,0.998908,1.0,0.705864,0.407523,0.553889,0.663207,0.515853,0.571824,0.998908,1,3,under_level_approvals (审批人级别低于规定的次数) = 2; activity_density (有付款月数 ÷ 合作月数) = ...
TP-B1885BF8,1.232356,0.998522,1.0,0.664420,0.435236,0.543333,0.755770,0.490476,0.586377,0.998522,1,4,under_level_approvals (审批人级别低于规定的次数) = 7; sensitive_expense_share (敏感费用比例（佣金...
TP-D6CA8844,1.214077,0.998420,1.0,0.694679,0.430969,0.608611,0.764098,0.573200,0.609161,0.998420,1,5,under_level_approvals (审批人级别低于规定的次数) = 3; sensitive_expense_share (敏感费用比例（佣金...
TP-E4DEA4DF,1.239347,0.998408,1.0,0.769202,0.488869,0.608611,0.650686,0.618616,0.650896,0.998408,1,6,under_level_approvals (审批人级别低于规定的次数) = 1; n_institution_links (关联机构数) = 8; s...
TP-907B6E23,1.170022,0.998357,1.0,0.697107,0.439321,0.575278,0.746001,0.579011,0.637013,0.998357,1,7,under_level_approvals (审批人级别低于规定的次数) = 3; sensitive_expense_share (敏感费用比例（佣金...
TP-2FCE3667,1.143218,0.997729,1.0,0.703096,0.493230,0.615833,0.647159,0.609775,0.621214,0.997729,1,8,under_level_approvals (审批人级别低于规定的次数) = 1; n_institution_links (关联机构数) = 2; s...
TP-67809112,1.111589,0.996146,1.0,0.668741,0.440057,0.557778,0.727168,0.556706,0.629651,0.996146,1,9,under_level_approvals (审批人级别低于规定的次数) = 3; sensitive_expense_share (敏感费用比例（佣金...
